In [1]:
# Cell 1: Cài đặt thư viện cần thiết cho InsightFace CUDA & PyTorch Geometric trên Kaggle / Google Colab
import os, sys, platform, subprocess

if platform.system() == "Linux":
    # Cài đặt các gói hệ thống cần thiết cho OpenCV và ONNX Runtime
    print("Đang cấu hình gói hệ thống trên Linux...")
    subprocess.run(
        "apt-get update -qq && apt-get install -y -qq libgl1 libglib2.0-0 >/dev/null 2>&1",
        shell=True
    )

# Cài đặt các thư viện Python: InsightFace, ONNX Runtime GPU, PyTorch Geometric, v.v.
print("Đang cài đặt các thư viện Python (InsightFace CUDA, PyG, OpenCV, ONNX Runtime GPU)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts",
    "torch-geometric", "insightface", "onnxruntime-gpu==1.21.0", "opencv-python", "tensorboard", "tqdm", "pyyaml"
])
print("Cài đặt hoàn tất! Môi trường đã sẵn sàng.")

Đang cấu hình gói hệ thống trên Linux...
Đang cài đặt các thư viện Python (InsightFace CUDA, PyG, OpenCV, ONNX Runtime GPU)...
Cài đặt hoàn tất! Môi trường đã sẵn sàng.


In [5]:
import kagglehub

# Download latest version
path_cache = kagglehub.dataset_download("nyvantran6634/cache-dataset-processed-v3", output_dir="./kaggle/cache-dataset-processed-v3")
path_origin = kagglehub.dataset_download("nyvantran6634/data-processed-v3", output_dir="./kaggle/data-processed-v3")

print("path_cache to dataset files:", path_cache)
print("path_origin to dataset files:", path_origin)

Using Colab cache for faster access to the 'cache-dataset-processed-v3' dataset.


100%|██████████| 15.6G/15.6G [01:43<00:00, 162MB/s] 

Extracting files...


path_cache to dataset files: /kaggle/input/cache-dataset-processed-v3
path_origin to dataset files: ./kaggle/data-processed-v3


In [3]:
# Cell 2: Import các thư viện cần thiết và cấu hình môi trường ONNX Runtime CUDA
from __future__ import annotations
import csv
import dataclasses
import json
import logging
import math
import os
import random
import sys
import tempfile
import time
import warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional

# Đảm bảo mã hóa UTF-8 trên console
if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass
if hasattr(sys.stderr, "reconfigure"):
    try:
        sys.stderr.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch_geometric.nn import DenseGCNConv, DenseGATConv
from torch_geometric.utils import to_undirected, add_self_loops, to_dense_adj
from tqdm import tqdm

# Cấu hình nạp DLL PyTorch CUDA và môi trường cho ONNX Runtime trên Windows / Linux
torch_lib_path = os.path.join(os.path.dirname(torch.__file__), "lib")
if os.path.exists(torch_lib_path):
    try:
        os.add_dll_directory(torch_lib_path)
    except Exception:
        pass
    os.environ["PATH"] = torch_lib_path + os.pathsep + os.environ.get("PATH", "")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

logger = logging.getLogger("DriverGuardian.GCN")

In [ ]:
# Cell 3: Hệ thống cấu hình trung tâm (config.py)
@dataclass
class TrainConfig:
    """
    Cấu hình tổng hợp toàn hệ thống Driver Guardian GCN.
    Tất cả các siêu tham số được quản lý tập trung và phân nhóm rõ ràng.
    """
    # =====================================================================
    # 1. CẤU HÌNH ĐỒ THỊ VÀ TRÍCH XUẤT ĐỘNG HỌC (GRAPH & KINEMATICS)
    # =====================================================================
    num_nodes: int = 68
    nose_idx: int = 30                 # Mốc tâm chóp mũi (gốc tọa độ cục bộ)
    left_eye_idx: int = 36             # Khóe mắt ngoài trái (đo IOD)
    right_eye_idx: int = 45            # Khóe mắt ngoài phải (đo IOD)
    eps: float = 1e-6                  # Tránh chia cho 0 khi tính khoảng cách IOD
    adaptive_init_scale: float = 1e-5  # Hệ số khởi tạo ma trận kề thích ứng

    # Cặp chỉ số đo độ mở mắt (EAR) & miệng (MAR)
    vertical_pairs: List[Tuple[int, int]] = field(default_factory=lambda: [
        (37, 41), (38, 40), (43, 47), (44, 46),  # Mí mắt trên - dưới
        (50, 58), (51, 57), (52, 56), (61, 67), (62, 66), (63, 65)  # Môi trên - dưới
    ])

    # Cặp chỉ số tiên nghiệm liên vùng (Mắt <-> Miệng, Mũi <-> Cằm)
    cross_region_pairs: List[Tuple[int, int]] = field(default_factory=lambda: [
        (37, 48), (41, 48), (38, 62), (40, 66),  # Mắt trái <-> Miệng
        (44, 54), (46, 54), (43, 62), (47, 66),  # Mắt phải <-> Miệng
        (30, 8), (30, 36), (30, 45), (27, 8)     # Chóp mũi <-> Cằm & Đuôi mắt
    ])

    # =====================================================================
    # 2. CẤU HÌNH MÔ HÌNH (MODEL ARCHITECTURE)
    # =====================================================================
    model_name: str = "stgcn"                 # "stgcn" | "transformer"
    in_channels: int = 6                      # [x, y, dx, dy, d2x, d2y]
    num_classes: int = 2                      # Mặc định 2 lớp (Alert vs Drowsy)

    # Siêu tham số riêng cho Model 1: DriverSTGCN (với AttentiveDrowsinessOutputHead)
    num_domains: int = 0                      # > 0 để bật Domain Generalization (MSTGCN)
    enable_regression: bool = False           # True để bật dự đoán điểm buồn ngủ liên tục
    stgcn_hidden_dims: List[int] = field(default_factory=lambda: [64, 128, 256])
    stgcn_strides: List[int] = field(default_factory=lambda: [1, 2, 2])
    temporal_kernel: int = 5                  # Độ dài kernel tích chập thời gian
    gat_heads: int = 4                        # Số attention heads của DenseGATConv
    dropout: float = 0.35                     # Tỷ lệ Dropout cho STGCN
    classifier_hidden_dim: Optional[int] = None  # None: mặc định curr_dim // 2

    # Siêu tham số riêng cho Model 2: DriverSTGCNTransformer
    d_model: int = 128                        # Kích thước biểu diễn của Transformer
    nhead: int = 4                            # Số Attention Heads trong Transformer
    num_transformer_layers: int = 2           # Số lớp TransformerEncoder
    max_len: int = 300                        # Chiều dài chuỗi thời gian tối đa
    transformer_dropout: float = 0.2
    stem_channels: List[int] = field(default_factory=lambda: [64, 128])
    stem_strides: List[int] = field(default_factory=lambda: [1, 2])
    dim_feedforward: Optional[int] = None     # None: mặc định d_model * 2
    init_std: float = 0.02                    # Độ lệch chuẩn khởi tạo CLS token & PosEmbedding

    # =====================================================================
    # 3. CẤU HÌNH BỘ DỮ LIỆU, INSIGHTFACE & CACHE (DATASET, INSIGHTFACE & CACHE)
    # =====================================================================
    dataset_dir: str = r"./kaggle/data-processed-v3/data_processed/"
    manifest_filename: str = "dataset_merged_split.csv"
    time_interval: float = 0.1                # Khoảng lấy mẫu thời gian thực (0.1s = 10 FPS; 0: đọc mọi frame)
    window_sec: Optional[float] = 10.0        # Cửa sổ thời gian (giây), None: toàn bộ video
    landmark_model_path: Optional[str] = None # None: tự tìm hoặc nạp mặc định của InsightFace
    landmark_model_name: str = "buffalo_l"    # Model InsightFace 68 3D landmarks
    det_size: Tuple[int, int] = (320, 320)    # Kích thước đầu vào SCRFD (320x320 hoặc 640x640)
    class_names: Dict[int, str] = field(default_factory=lambda: {
        0: "Alert",
        1: "Drowsy"
    })
    letterbox_size: int | Tuple[int, int] = (640, 480)  # Kích thước (w, h) hoặc int S; (0, 0) / 0 để tắt

    # Cấu hình trích xuất & nạp cache landmark (.pt)
    use_cache: bool = True                    # Bật nạp chuỗi landmark từ file {cache_dir}/{split}.pt trong GCNDataset
    cache_dir: str = "/kaggle/input/cache-dataset-processed-v3/cache_landmarks"  # Thư mục chứa train.pt, val.pt, test.pt
    cache_splits: List[str] = field(default_factory=lambda: ["train", "val", "test"])
    landmark_batch_size: int = 64             # Batch size suy luận 1k3d68.onnx (Batched GPU Inference)
    extract_num_workers: int = 4              # Số luồng CPU giải mã video song song khi tạo cache
    extract_max_side: int = 0                 # Thu nhỏ frame khi tạo cache (nếu tắt letterbox)
    cache_save_every: int = 100               # Lưu tạm (atomic save) file .pt sau mỗi N video
    resume_cache: bool = True                 # Tiếp tục từ các video đã có trong file .pt

    # =====================================================================
    # 4. CẤU HÌNH DATALOADER & HUẤN LUYỆN (DATALOADER & TRAINING)
    # =====================================================================
    batch_size: int = 32
    num_workers: int = 0
    pin_memory: bool = True
    shuffle_train: bool = True

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    lr: float = 1e-3
    weight_decay: float = 1e-4
    epochs: int = 20
    warmup_epochs: int = 5
    min_lr_ratio: float = 0.05
    max_grad_norm: float = 1.0
    amp: bool = True
    label_smoothing: float = 0.0             # Hệ số làm mịn nhãn Label Smoothing (0.0: tắt)
    save_dir: str = "GCN/checkpoints"
    log_dir: Optional[str] = None
    resume: Optional[str] = None
    seed: int = 42

    def __post_init__(self):
        """Tự động điều chỉnh và đồng bộ các thuộc tính phụ thuộc."""
        if not (0.0 <= float(self.label_smoothing) < 1.0):
            raise ValueError(f"label_smoothing phải nằm trong khoảng [0.0, 1.0), nhận được: {self.label_smoothing}")

        # Chuẩn hóa letterbox_size về tuple (w, h) hoặc int tương thích
        if isinstance(self.letterbox_size, int):
            self.letterbox_size = (self.letterbox_size, self.letterbox_size) if self.letterbox_size > 0 else (0, 0)
        elif isinstance(self.letterbox_size, (list, tuple)) and len(self.letterbox_size) == 2:
            w, h = int(self.letterbox_size[0]), int(self.letterbox_size[1])
            self.letterbox_size = (w, h) if (w > 0 and h > 0) else (0, 0)
        else:
            self.letterbox_size = (0, 0)

        if isinstance(self.det_size, (list, tuple)) and len(self.det_size) == 2:
            self.det_size = (int(self.det_size[0]), int(self.det_size[1]))

        if self. vertical_pairs is not None:
            self.vertical_pairs = [tuple(p) for p in self.vertical_pairs]
        if self.cross_region_pairs is not None:
            self.cross_region_pairs = [tuple(p) for p in self.cross_region_pairs]

        if self.class_names is None or len(self.class_names) != self.num_classes:
            if self.num_classes == 2:
                self.class_names = {0: "Alert", 1: "Drowsy"}
            elif self.num_classes == 3:
                self.class_names = {0: "Alert", 1: "Fatigue", 2: "Drowsy"}
            else:
                self.class_names = {i: f"Class_{i}" for i in range(self.num_classes)}

    def to_dict(self) -> Dict[str, Any]:
        """Chuyển đổi toàn bộ cấu hình sang dictionary."""
        return asdict(self)

    def save(self, path: str | Path) -> None:
        """Lưu cấu hình sang tệp JSON hoặc YAML (nếu có PyYAML)."""
        p = Path(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        data = self.to_dict()

        if p.suffix.lower() in (".yaml", ".yml"):
            try:
                import yaml
                with open(p, "w", encoding="utf-8") as f:
                    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)
                return
            except ImportError:
                pass  # Fallback to json if PyYAML is not installed

        with open(p, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4, ensure_ascii=False)

    @classmethod
    def load(cls, path: str | Path) -> "TrainConfig":
        """Nạp cấu hình từ tệp JSON hoặc YAML (tương thích ngược với các phiên bản cũ/mới)."""
        p = Path(path)
        if not p.is_file():
            raise FileNotFoundError(f"Không tìm thấy file config tại: {p}")

        content = None
        if p.suffix.lower() in (".yaml", ".yml"):
            try:
                import yaml
                with open(p, "r", encoding="utf-8") as f:
                    content = yaml.safe_load(f)
            except ImportError:
                pass

        if content is None:
            with open(p, "r", encoding="utf-8") as f:
                content = json.load(f)

        # Chuyển class_names key sang int nếu nạp từ json (json lưu key dạng string)
        if "class_names" in content and isinstance(content["class_names"], dict):
            content["class_names"] = {int(k): v for k, v in content["class_names"].items()}

        valid_fields = {f.name for f in dataclasses.fields(cls)}
        filtered_content = {k: v for k, v in content.items() if k in valid_fields}
        return cls(**filtered_content)

    # -----------------------------------------------------------------
    # Helper methods trích xuất kwargs để truyền vào các components
    # -----------------------------------------------------------------
    def get_stgcn_kwargs(self) -> Dict[str, Any]:
        """Trích xuất tham số thuần túy cho mô hình DriverSTGCN."""
        return {
            "in_channels": self.in_channels,
            "num_classes": self.num_classes,
            "num_nodes": self.num_nodes,
            "num_domains": self.num_domains,
            "enable_regression": self.enable_regression,
            "hidden_dims": list(self.stgcn_hidden_dims),
            "strides": list(self.stgcn_strides),
            "dropout": self.dropout,
            "temporal_kernel": self.temporal_kernel,
            "gat_heads": self.gat_heads,
            "classifier_hidden_dim": self.classifier_hidden_dim,
            "nose_idx": self.nose_idx,
            "left_eye_idx": self.left_eye_idx,
            "right_eye_idx": self.right_eye_idx,
            "eps": self.eps,
            "adaptive_init_scale": self.adaptive_init_scale,
            "vertical_pairs": list(self.vertical_pairs) if self.vertical_pairs is not None else None,
            "cross_region_pairs": list(self.cross_region_pairs) if self.cross_region_pairs is not None else None,
        }

    def get_transformer_kwargs(self) -> Dict[str, Any]:
        """Trích xuất tham số thuần túy cho mô hình DriverSTGCNTransformer."""
        return {
            "in_channels": self.in_channels,
            "num_classes": self.num_classes,
            "num_nodes": self.num_nodes,
            "d_model": self.d_model,
            "nhead": self.nhead,
            "num_transformer_layers": self.num_transformer_layers,
            "max_len": self.max_len,
            "dropout": self.transformer_dropout,
            "stem_channels": list(self.stem_channels),
            "stem_strides": list(self.stem_strides),
            "dim_feedforward": self.dim_feedforward,
            "init_std": self.init_std,
            "nose_idx": self.nose_idx,
            "left_eye_idx": self.left_eye_idx,
            "right_eye_idx": self.right_eye_idx,
            "eps": self.eps,
            "adaptive_init_scale": self.adaptive_init_scale,
            "vertical_pairs": list(self.vertical_pairs) if self.vertical_pairs is not None else None,
            "cross_region_pairs": list(self.cross_region_pairs) if self.cross_region_pairs is not None else None,
        }

    def get_extractor_kwargs(self) -> Dict[str, Any]:
        """Trích xuất tham số thuần túy cho FaceLandmarkExtractor."""
        return {
            "model_path": self.landmark_model_path,
            "time_interval": self.time_interval,
            "device": self.device,
            "det_size": self.det_size,
            "model_name": self.landmark_model_name,
            "landmark_batch_size": self.landmark_batch_size,
            "letterbox_size": self.letterbox_size,
        }

    def get_dataset_kwargs(self, split: str = "train") -> Dict[str, Any]:
        """Trích xuất tham số thuần túy cho GCNDataset."""
        return {
            "dataset_dir": self.dataset_dir,
            "split": split,
            "time_interval": self.time_interval,
            "window_sec": self.window_sec,
            "manifest_filename": self.manifest_filename,
            "landmark_model_path": self.landmark_model_path,
            "letterbox_size": self.letterbox_size,
            "device": self.device,
            "det_size": self.det_size,
            "model_name": self.landmark_model_name,
            "use_cache": self.use_cache,
            "cache_dir": self.cache_dir,
            "landmark_batch_size": self.landmark_batch_size,
        }

    def get_dataloader_kwargs(self) -> Dict[str, Any]:
        """Trích xuất tham số thuần túy cho build_gcn_dataloaders."""
        return {
            "dataset_dir": self.dataset_dir,
            "batch_size": self.batch_size,
            "num_workers": self.num_workers,
            "time_interval": self.time_interval,
            "window_sec": self.window_sec,
            "manifest_filename": self.manifest_filename,
            "pin_memory": self.pin_memory,
            "shuffle_train": self.shuffle_train,
            "landmark_model_path": self.landmark_model_path,
            "letterbox_size": self.letterbox_size,
            "device": self.device,
            "det_size": self.det_size,
            "model_name": self.landmark_model_name,
            "use_cache": self.use_cache,
            "cache_dir": self.cache_dir,
            "landmark_batch_size": self.landmark_batch_size,
        }


# =====================================================================
# HÀM TIỆN ÍCH CẤP MODULE
# =====================================================================
DEFAULT_CONFIG = TrainConfig()


def get_default_config(num_classes: int = 2) -> TrainConfig:
    """Tạo đối tượng TrainConfig với số nhãn phân loại tùy chọn (2 hoặc 3 lớp)."""
    return TrainConfig(num_classes=num_classes)


def save_config(config: TrainConfig, path: str | Path) -> None:
    """Lưu đối tượng TrainConfig ra tệp JSON hoặc YAML."""
    config.save(path)


def load_config(path: str | Path) -> TrainConfig:
    """Nạp tệp cấu hình từ JSON hoặc YAML và trả về TrainConfig."""
    return TrainConfig.load(path)

In [6]:
# Cell 4: Dataset (nạp cache landmark .pt & cảnh báo sai lệch config) và Bộ trích xuất 68 điểm mốc khuôn mặt InsightFace CUDA (dataset.py)
# =====================================================================
# 1. BẢNG CHỈ SỐ ÁNH XẠ 68 ĐIỂM MỐC (MediaPipe 478 -> Dlib 68)
# Khớp chính xác với cấu trúc đồ thị FacialGraphBuilder trong block.py
# =====================================================================
LANDMARK_68_INDICES = [
    # Viền hàm (0-16): 17 điểm
    162, 234, 93, 58, 172, 136, 149, 148, 152, 377, 378, 365, 397, 288, 323, 454, 389,
    # Lông mày trái (17-21) & phải (22-26): 10 điểm
    71, 63, 105, 66, 107, 336, 296, 334, 293, 301,
    # Sống mũi & chân mũi (27-35): 9 điểm
    168, 197, 5, 4, 75, 97, 2, 326, 305,
    # Mắt trái (36-41) & Mắt phải (42-47): 12 điểm
    33, 160, 158, 133, 153, 144, 362, 385, 387, 263, 373, 380,
    # Môi ngoài (48-59) & Môi trong (60-67): 20 điểm
    61, 39, 37, 0, 267, 269, 291, 405, 314, 17, 84, 181,
    78, 82, 13, 312, 308, 317, 14, 87,
]


def _parse_letterbox_size(new_size: Optional[int | Tuple[int, int] | List[int]]) -> Tuple[int, int]:
    """Chuẩn hóa tham số kích thước letterbox về bộ (target_w, target_h). Trả về (0, 0) nếu tắt."""
    if new_size is None:
        return (0, 0)
    if isinstance(new_size, int):
        return (new_size, new_size) if new_size > 0 else (0, 0)
    if isinstance(new_size, (tuple, list)) and len(new_size) == 2:
        w, h = int(new_size[0]), int(new_size[1])
        return (w, h) if (w > 0 and h > 0) else (0, 0)
    raise ValueError(f"new_size phải là int hoặc tuple/list (w, h), nhận được: {new_size}")


def letterbox(
        image: np.ndarray,
        new_size: Optional[int | Tuple[int, int] | List[int]] = (640, 480),
        color: Tuple[int, int, int] = (114, 114, 114),
) -> np.ndarray:
    """
    Resize ảnh giữ nguyên tỷ lệ khung hình gốc với đệm màu viền về kích thước đích (w, h).
    - image: Ảnh BGR numpy array shape (H, W, 3).
    - new_size: Kích thước đích theo (w, h) (vd: (640, 480)) hoặc số nguyên S tương ứng (S, S).
    - color: Màu đệm viền BGR, mặc định (114, 114, 114).
    Trả về ảnh numpy array có shape (target_h, target_w, 3).
    """
    target_w, target_h = _parse_letterbox_size(new_size)
    if target_w <= 0 or target_h <= 0:
        return image

    h, w = image.shape[:2]
    if w == target_w and h == target_h:
        return image

    scale = min(target_w / float(w), target_h / float(h))
    nw, nh = max(1, round(w * scale)), max(1, round(h * scale))
    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(image, (nw, nh), interpolation=interp)

    if nw == target_w and nh == target_h:
        return resized

    canvas = np.full((target_h, target_w, 3), color, dtype=image.dtype)
    x, y = (target_w - nw) // 2, (target_h - nh) // 2
    canvas[y:y + nh, x:x + nw] = resized
    return canvas


def read_video_frames(
        video_path: str | Path,
        time_interval: float = 0.1,
        window_sec: Optional[float] = None,
        letterbox_size: Optional[int | Tuple[int, int] | List[int]] = (640, 480),
        max_side: int = 0,
) -> Tuple[List[np.ndarray], float, int, float]:
    """
    Đọc frame từ video chỉ với MỘT lần mở cv2.VideoCapture.
    - time_interval > 0: lấy mẫu cách đều theo thời gian thực; == 0: đọc tuần tự mọi frame.
    - window_sec: None = toàn bộ video; số giây = chỉ lấy cửa sổ đầu tiên (bắt đầu từ 0s).
    - letterbox_size: (w, h) kích thước chuẩn hóa letterbox đồng nhất tỷ lệ cho mọi video.
    - max_side: tùy chọn tương thích ngược (nếu letterbox_size=(0, 0)).
    Trả về (frames, fps, total_frames, duration_sec).
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f"Không thể mở video: {video_path}")

    try:
        fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
        if not math.isfinite(fps) or fps <= 0:
            fps = 30.0
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0:
            raise IOError(f"Video không có frame hợp lệ: {video_path}")
        duration_sec = total_frames / fps
        win = duration_sec if window_sec is None else min(float(window_sec), duration_sec)

        if time_interval == 0:
            num = total_frames if window_sec is None else max(1, min(int(round(win * fps)), total_frames))
            indices = np.arange(num, dtype=int)
        else:
            timestamps = np.arange(0, win, time_interval)
            if len(timestamps) == 0:
                timestamps = np.array([0.0])
            indices = np.clip(np.rint(timestamps * fps).astype(int), 0, total_frames - 1)

        target_w, target_h = _parse_letterbox_size(letterbox_size)
        frames: List[np.ndarray] = []
        curr_frame = 0
        for target_idx in indices:
            if target_idx < curr_frame and frames:
                frames.append(frames[-1])  # Chỉ số trùng lặp (time_interval < 1/fps)
                continue
            while curr_frame < target_idx:
                if not cap.grab():
                    break
                curr_frame += 1
            ret, frame = cap.read()
            if not ret:
                break
            if target_w > 0 and target_h > 0:
                frame = letterbox(frame, new_size=(target_w, target_h))
            elif max_side and max(frame.shape[:2]) > max_side:
                scale = max_side / float(max(frame.shape[:2]))
                frame = cv2.resize(
                    frame,
                    (max(1, round(frame.shape[1] * scale)), max(1, round(frame.shape[0] * scale))),
                    interpolation=cv2.INTER_AREA,
                )
            frames.append(frame)
            curr_frame += 1
    finally:
        cap.release()

    if not frames:
        raise IOError(f"Không đọc được frame nào từ video: {video_path}")
    # Bù frame cuối nếu video kết thúc sớm hơn dự kiến
    if len(frames) < len(indices):
        frames.extend([frames[-1]] * (len(indices) - len(frames)))
    return frames, fps, total_frames, duration_sec


# =====================================================================
# 2. BỘ TRÍCH XUẤT 68 ĐIỂM MỐC KHUÔN MẶT BẰNG INSIGHTFACE (CUDA, BATCHED)
# =====================================================================
class FaceLandmarkExtractor:
    """
    Trích xuất 68 điểm mốc 3D (lấy x, y) bằng InsightFace, tối ưu tốc độ:
      - Chỉ nạp det_*.onnx (SCRFD) + 1k3d68.onnx, không nạp recognition / genderage / 2d106det.
      - max_num=1 (metric="max"): chỉ giữ khuôn mặt lớn nhất trước khi chạy 1k3d68.
      - Chạy 1k3d68.onnx theo lô (batched) cho toàn bộ frame hợp lệ của video.
      - Chỉ lấy 68 điểm cuối, biến đổi ngược Affine vector hóa bằng np.einsum, bỏ qua 3D Pose (lstsq).
    Tọa độ chuẩn hóa theo kích thước (w, h) của frame, kết quả trong [0, 1].
    """

    def __init__(
            self,
            model_path: Optional[str | Path] = None,
            time_interval: float = 0.0,
            device: str = "cuda",
            det_size: Tuple[int, int] = (320, 320),
            model_name: str = "buffalo_l",
            landmark_batch_size: int = 64,
            det_thresh: float = 0.5,
            letterbox_size: Optional[int | Tuple[int, int] | List[int]] = (640, 480),
    ):
        self.model_path = Path(model_path) if model_path is not None else None
        self.time_interval = float(time_interval)
        self.device = device.lower().strip()
        self.det_size = tuple(det_size)
        self.model_name = model_name
        self.landmark_batch_size = max(1, int(landmark_batch_size))
        self.det_thresh = det_thresh
        self.letterbox_size = _parse_letterbox_size(letterbox_size)
        self.det = None
        self.lmk = None

    def _init_landmarker(self):
        """Khởi tạo trễ (lazy init) chỉ 2 mô hình ONNX cần thiết để tương thích đa tiến trình."""
        import glob
        import onnxruntime as ort
        from insightface.model_zoo import model_zoo
        from insightface.utils import ensure_available

        ort.set_default_logger_severity(3)

        available_providers = ort.get_available_providers()
        use_cuda = self.device.startswith("cuda") and ("CUDAExecutionProvider" in available_providers)

        if self.device.startswith("cuda") and not use_cuda:
            logger.warning(
                f"[WARNING] Không tìm thấy CUDAExecutionProvider (khả dụng: {available_providers})! "
                "Mô hình InsightFace sẽ chạy trên CPU."
            )

        if use_cuda:
            providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
            ctx_id = 0
        else:
            providers = ["CPUExecutionProvider"]
            ctx_id = -1

        root = str(self.model_path) if (self.model_path is not None and self.model_path.is_dir()) else "~/.insightface"
        model_dir = ensure_available("models", self.model_name, root=root)

        det_files = sorted(glob.glob(os.path.join(model_dir, "det_*.onnx")))
        lmk_file = os.path.join(model_dir, "1k3d68.onnx")
        if not det_files or not os.path.isfile(lmk_file):
            raise FileNotFoundError(
                f"Thiếu det_*.onnx hoặc 1k3d68.onnx trong {model_dir} (model_name='{self.model_name}')."
            )

        self.det = model_zoo.get_model(det_files[0], providers=providers)
        self.lmk = model_zoo.get_model(lmk_file, providers=providers)
        self.det.prepare(ctx_id, input_size=self.det_size, det_thresh=self.det_thresh)
        self.lmk.prepare(ctx_id)
        if hasattr(self.lmk, "require_pose"):
            self.lmk.require_pose = False
        logger.info(
            f"InsightFace ({self.model_name}) khởi tạo thành công trên provider: {providers[0]} "
            f"(det={os.path.basename(det_files[0])}, det_size={self.det_size}, "
            f"landmark_batch={self.landmark_batch_size}, letterbox={self.letterbox_size})."
        )

    def release(self) -> None:
        """Giải phóng session ONNX và bộ nhớ GPU."""
        import gc
        self.det = None
        self.lmk = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def extract(self, frames: List[np.ndarray]) -> np.ndarray:
        """
        Nhận danh sách frames BGR và trích xuất chuỗi 68 điểm mốc chuẩn hóa trong [0, 1].
        Trả về numpy array kích thước: (T, 68, 2)
        """
        if self.det is None or self.lmk is None:
            self._init_landmarker()

        from insightface.utils import face_align

        T = len(frames)
        if T == 0:
            return np.zeros((0, 68, 2), dtype=np.float32)

        target_w, target_h = self.letterbox_size
        if target_w > 0 and target_h > 0:
            processed_frames = []
            for f in frames:
                if f.shape[1] != target_w or f.shape[0] != target_h:
                    processed_frames.append(letterbox(f, new_size=(target_w, target_h)))
                else:
                    processed_frames.append(f)
            frames = processed_frames

        # 1. Phát hiện khuôn mặt lớn nhất trên từng frame
        bboxes: List[Optional[np.ndarray]] = []
        for frame in frames:
            det, _ = self.det.detect(frame, max_num=1, metric="max")
            bboxes.append(det[0, :4] if (det is not None and det.shape[0] > 0) else None)

        valid_idx = [i for i, b in enumerate(bboxes) if b is not None]
        lm68 = np.zeros((T, 68, 2), dtype=np.float32)
        found = np.zeros(T, dtype=bool)

        # 2. Chạy 1k3d68.onnx theo lô trên các frame có khuôn mặt
        isz = int(self.lmk.input_size[0])
        mean = float(self.lmk.input_mean)
        std = float(self.lmk.input_std)
        for s in range(0, len(valid_idx), self.landmark_batch_size):
            chunk = valid_idx[s:s + self.landmark_batch_size]
            aimgs, ims = [], []
            for i in chunk:
                bbox = bboxes[i]
                w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
                center = (bbox[2] + bbox[0]) / 2, (bbox[3] + bbox[1]) / 2
                scale = isz / (max(w, h) * 1.5)
                aimg, M = face_align.transform(frames[i], center, isz, scale, 0)
                aimgs.append(aimg)
                ims.append(cv2.invertAffineTransform(M))

            blob = cv2.dnn.blobFromImages(aimgs, 1.0 / std, (isz, isz), (mean, mean, mean), swapRB=True)
            pred = self.lmk.session.run(self.lmk.output_names, {self.lmk.input_name: blob})[0]
            pred = pred.reshape(len(chunk), -1, 3)[:, -68:, :2].astype(np.float32)
            pred = (pred + 1.0) * (isz // 2)

            homo = np.concatenate([pred, np.ones((len(chunk), 68, 1), dtype=np.float32)], axis=-1)  # (n, 68, 3)
            im_batch = np.stack(ims).astype(np.float32)  # (n, 2, 3)
            out = np.einsum("nkj,nij->nki", homo, im_batch)  # (n, 68, 2)

            for j, i in enumerate(chunk):
                fh, fw = frames[i].shape[:2]
                lm68[i, :, 0] = out[j, :, 0] / float(max(1, fw))
                lm68[i, :, 1] = out[j, :, 1] / float(max(1, fh))
                found[i] = True

        # 3. Điền các frame mất mặt: forward-fill, sau đó backward-fill các frame đầu
        if found.any():
            last = None
            for i in range(T):
                if found[i]:
                    last = lm68[i]
                elif last is not None:
                    lm68[i] = last
            first = int(np.argmax(found))
            if first > 0:
                lm68[:first] = lm68[first]

        return lm68

    def extract_from_video(
            self,
            video_path: str | Path,
            time_interval: Optional[float] = None,
            window_sec: Optional[float] = None,
            letterbox_size: Optional[int | Tuple[int, int] | List[int]] = None,
    ) -> Tuple[np.ndarray, List[np.ndarray], Dict[str, Any]]:
        """
        Đọc frame trực tiếp từ video file và trích xuất chuỗi 68 landmark.
        - time_interval: khoảng thời gian (giây) giữa 2 frame lấy mẫu (0: đọc mọi frame, None: dùng self.time_interval).
        - window_sec: thời lượng cửa sổ (giây) cần lấy. None = toàn bộ video.
        - letterbox_size: (w, h) kích thước chuẩn hóa letterbox (None = dùng self.letterbox_size).
        """
        ti = self.time_interval if time_interval is None else float(time_interval)
        if ti < 0:
            raise ValueError(f"time_interval phải là số không âm (>= 0): {ti}")

        lbs = self.letterbox_size if letterbox_size is None else _parse_letterbox_size(letterbox_size)
        frames, fps, total_frames, duration_sec = read_video_frames(
            video_path, ti, window_sec, letterbox_size=lbs
        )
        lm68 = self.extract(frames)
        meta = {
            "video_path": str(video_path),
            "fps": fps,
            "total_frames": total_frames,
            "sampled_frames": len(frames),
            "duration_sec": duration_sec,
            "time_interval": ti,
            "letterbox_size": lbs,
        }
        return lm68, frames, meta


# =====================================================================
# 3. LỚP GCNDATASET (HỖ TRỢ CACHE .PT & TRÍCH XUẤT TRỰC TIẾP TỪ VIDEO)
# =====================================================================
class GCNDataset(Dataset):
    """
    Dataset đọc chuỗi 68 điểm mốc khuôn mặt cho ST-GCN, hỗ trợ:
      - Nạp trực tiếp từ file cache .pt ({cache_dir}/{split}.pt) vào RAM và cắt cửa sổ thời gian (Random Crop cho Train, Center Crop cho Val/Test).
      - Fallback đọc video trực tiếp từ thư mục dữ liệu khi use_cache=False, return_frames=True, hoặc thiếu mẫu trong cache.
      - Từng video có FPS riêng (60, 30, 25 FPS, v.v.).
      - time_interval = 0 -> đọc frame tuần tự thông thường từ video (không bỏ cách frame).
      - time_interval > 0 -> cơ chế lấy mẫu sau mỗi time_interval giây (time-domain sampling).
      - window_sec = None -> lấy frame trên toàn bộ thời lượng video.
      - window_sec = số giây -> lấy frame trong phạm vi cửa sổ đó.
    """

    def __init__(
            self,
            dataset_dir: str | Path = r"E:\LSTM\data_processed",
            split: str = "train",
            time_interval: float = 0.1,
            window_sec: Optional[float] = 10.0,
            manifest_filename: str = "dataset_merged_split.csv",
            landmark_model_path: Optional[str | Path] = None,
            letterbox_size: int | Tuple[int, int] = (640, 480),
            return_frames: bool = False,
            device: str = "cuda",
            det_size: Tuple[int, int] = (320, 320),
            model_name: str = "buffalo_l",
            use_cache: bool = True,
            cache_dir: Optional[str | Path] = None,
            landmark_batch_size: int = 64,
    ):
        if split not in ("train", "val", "test"):
            raise ValueError(f"Split không hợp lệ: {split}. Chỉ chấp nhận 'train', 'val', 'test'.")

        self.manifest_filename = manifest_filename

        if time_interval < 0:
            raise ValueError(f"time_interval phải là số không âm (>= 0): {time_interval}")
        if window_sec is not None and window_sec <= 0:
            raise ValueError(f"window_sec phải là số dương hoặc None: {window_sec}")

        self.root = Path(dataset_dir)
        self.split = split
        self.time_interval = float(time_interval)
        self.cache_time_interval: Optional[float] = None
        self.window_sec = float(window_sec) if window_sec is not None else None
        self.letterbox_size = _parse_letterbox_size(letterbox_size)
        self.return_frames = return_frames
        self.device = device
        self.det_size = tuple(det_size)
        self.model_name = model_name
        self.use_cache = bool(use_cache)
        self.cache_dir = Path(cache_dir) if cache_dir else None
        self.cached: Optional[Dict[str, Dict[str, Any]]] = None

        self.extractor = FaceLandmarkExtractor(
            model_path=landmark_model_path,
            time_interval=self.time_interval,
            device=self.device,
            det_size=self.det_size,
            model_name=self.model_name,
            landmark_batch_size=landmark_batch_size,
            letterbox_size=self.letterbox_size,
        )
        self.samples = self._load_samples()
        self._load_cache()

        if len(self.samples) == 0:
            logger.warning(f"Không tìm thấy mẫu nào trong {self.root} cho split={split}")

    def _load_samples(self) -> List[Dict[str, Any]]:
        """Nạp danh sách video từ manifest CSV hoặc quét thư mục dự phòng."""
        manifest = self.root / self.manifest_filename
        samples = []

        if manifest.is_file():
            with manifest.open(encoding="utf-8-sig", newline="") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    if row.get("split", "").strip().lower() == self.split:
                        rel_path = row.get("rel_path") or row.get("processed_path")
                        if not rel_path:
                            continue
                        norm_key = Path(rel_path.replace("\\", "/")).as_posix()
                        samples.append({
                            "key": norm_key,
                            "path": self.root / norm_key,
                            "label": int(row["label"]),
                            "video_id": row.get("video_id") or row.get("clip_id") or Path(rel_path).stem,
                            "fps": float(row.get("fps", 0.0)),
                            "total_frames": int(float(row.get("frame_count", 0))),
                            "duration_sec": float(row.get("duration_sec", 0.0)),
                        })
        else:
            for label, folder in ((0, "0_alert"), (1, "1_drowsy")):
                split_dir = self.root / self.split / folder
                if split_dir.is_dir():
                    for p in sorted(split_dir.glob("*.mp4")):
                        samples.append({
                            "key": p.relative_to(self.root).as_posix(),
                            "path": p,
                            "label": label,
                            "video_id": p.stem,
                            "fps": 0.0,
                            "total_frames": 0,
                            "duration_sec": 0.0,
                        })
        return samples

    @staticmethod
    def _emit_cache_warning(msg: str) -> None:
        """Phát cảnh báo đồng thời qua logger.warning và warnings.warn (nếu logger chưa gắn handler)."""
        logger.warning(msg)
        if not logger.handlers and not logging.getLogger().handlers:
            warnings.warn(msg, UserWarning, stacklevel=2)

    def _check_cache_config_mismatches(self, cache_path: Path, payload: Dict[str, Any]) -> None:
        """
        Kiểm tra toàn diện sự sai lệch giữa metadata lưu trong file cache (.pt & config.json)
        với cấu hình hiện tại của GCNDataset / TrainConfig và phát cảnh báo chi tiết.
        """
        cache_cfg_data: Dict[str, Any] = {}
        if self.cache_dir is not None:
            cfg_json_path = self.cache_dir / "config.json"
            if cfg_json_path.is_file():
                try:
                    with open(cfg_json_path, "r", encoding="utf-8") as f:
                        loaded = json.load(f)
                        if isinstance(loaded, dict):
                            cache_cfg_data = loaded
                except Exception:
                    pass

        # 1. Kiểm tra split
        cache_split = payload.get("split")
        if cache_split is not None and str(cache_split).strip().lower() != self.split:
            self._emit_cache_warning(
                f"[CACHE MISMATCH] {cache_path.name}: split trong cache ('{cache_split}') "
                f"khác với split đang khởi tạo ('{self.split}')!"
            )

        # 2. Kiểm tra time_interval
        raw_ti = payload.get("time_interval", cache_cfg_data.get("time_interval", None))
        if raw_ti is not None:
            cache_ti = float(raw_ti)
            self.cache_time_interval = cache_ti if cache_ti >= 0 else self.time_interval
            if abs(cache_ti - self.time_interval) > 1e-6:
                self._emit_cache_warning(
                    f"[CACHE MISMATCH] {cache_path.name}: time_interval trong cache ({cache_ti}s) "
                    f"khác cấu hình hiện tại ({self.time_interval}s). "
                    f"Tự động đồng bộ bước cắt cửa sổ theo time_interval={cache_ti}s của cache."
                )
        else:
            self.cache_time_interval = self.time_interval

        # 3. Kiểm tra letterbox_size
        raw_lbs = payload.get("letterbox_size", cache_cfg_data.get("letterbox_size", None))
        if raw_lbs is not None:
            try:
                cache_lbs = _parse_letterbox_size(raw_lbs)
                if cache_lbs != self.letterbox_size:
                    self._emit_cache_warning(
                        f"[CACHE MISMATCH] {cache_path.name}: letterbox_size trong cache ({cache_lbs}) "
                        f"khác cấu hình hiện tại ({self.letterbox_size})."
                    )
            except Exception:
                pass

        # 4. Kiểm tra det_size
        raw_det = payload.get("det_size", cache_cfg_data.get("det_size", None))
        if raw_det is not None and isinstance(raw_det, (list, tuple)) and len(raw_det) == 2:
            cache_det = (int(raw_det[0]), int(raw_det[1]))
            if cache_det != tuple(self.det_size):
                self._emit_cache_warning(
                    f"[CACHE MISMATCH] {cache_path.name}: det_size trong cache ({cache_det}) "
                    f"khác cấu hình hiện tại ({self.det_size})."
                )

        # 5. Kiểm tra model_name (InsightFace)
        cache_model = payload.get("model_name", cache_cfg_data.get("landmark_model_name", None))
        if cache_model is not None and str(cache_model).strip() != str(self.model_name).strip():
            self._emit_cache_warning(
                f"[CACHE MISMATCH] {cache_path.name}: landmark model_name trong cache ('{cache_model}') "
                f"khác cấu hình hiện tại ('{self.model_name}')."
            )

        # 6. Kiểm tra manifest_filename từ config.json của cache (nếu có)
        cache_manifest = cache_cfg_data.get("manifest_filename", None)
        if cache_manifest is not None and str(cache_manifest).strip() != str(self.manifest_filename).strip():
            self._emit_cache_warning(
                f"[CACHE MISMATCH] {cache_path.name}: manifest_filename khi tạo cache ('{cache_manifest}') "
                f"khác cấu hình hiện tại ('{self.manifest_filename}')."
            )

    def _load_cache(self) -> None:
        """Nạp chuỗi landmark toàn thời lượng từ {cache_dir}/{split}.pt vào RAM và kiểm tra sai lệch cấu hình."""
        if not self.use_cache or self.cache_dir is None:
            return

        cache_path = self.cache_dir / f"{self.split}.pt"
        if not cache_path.is_file():
            self._emit_cache_warning(
                f"[CACHE] Không tìm thấy {cache_path}. Sẽ fallback trích xuất trực tiếp từ video."
            )
            return

        try:
            try:
                payload = torch.load(cache_path, map_location="cpu", weights_only=True)
            except Exception:
                payload = torch.load(cache_path, map_location="cpu", weights_only=False)
        except TypeError:
            payload = torch.load(cache_path, map_location="cpu")

        if isinstance(payload, dict) and "samples" in payload:
            self._check_cache_config_mismatches(cache_path, payload)
            self.cached = payload["samples"]
        elif isinstance(payload, dict):
            self.cache_time_interval = self.time_interval
            self.cached = payload
        else:
            self._emit_cache_warning(
                f"[CACHE] Định dạng dữ liệu trong {cache_path} không hợp lệ ({type(payload)}). Bỏ qua cache."
            )
            return

        hit = 0
        label_mismatches = 0
        for s in self.samples:
            key = s["key"]
            if key in self.cached:
                hit += 1
                cached_item = self.cached[key]
                if isinstance(cached_item, dict) and "label" in cached_item:
                    if int(cached_item["label"]) != int(s["label"]):
                        label_mismatches += 1

        logger.info(
            f"[CACHE] Đã nạp {cache_path.name}: {len(self.cached)} video trong cache | "
            f"khớp {hit}/{len(self.samples)} mẫu của split '{self.split}'."
        )
        if hit < len(self.samples):
            self._emit_cache_warning(
                f"[CACHE MISMATCH] Thiếu {len(self.samples) - hit}/{len(self.samples)} mẫu của split '{self.split}' "
                f"trong {cache_path.name}; các mẫu thiếu sẽ được trích xuất trực tiếp từ video."
            )
        if label_mismatches > 0:
            self._emit_cache_warning(
                f"[CACHE MISMATCH] Phát hiện {label_mismatches} mẫu trong {cache_path.name} có nhãn 'label' "
                f"khác với manifest hiện tại ('{self.manifest_filename}')!"
            )

    def _slice_temporal_window(self, full_lm: torch.Tensor, fps: float, duration_sec: float) -> torch.Tensor:
        """
        Cắt cửa sổ window_sec trên chuỗi landmark toàn thời lượng (2, T_full, 68).
        Train: vị trí bắt đầu ngẫu nhiên mỗi lần gọi (Random Temporal Crop); Val/Test: cắt chính giữa.
        """
        T = full_lm.shape[1]
        if self.window_sec is None or T <= 1:
            return full_lm.clone()

        eff_ti = self.cache_time_interval if self.cache_time_interval is not None else self.time_interval
        win = min(self.window_sec, duration_sec) if duration_sec > 0 else self.window_sec
        if eff_ti > 0:
            win_frames = len(np.arange(0, win, eff_ti))
        else:
            win_frames = int(round(win * (fps if fps > 0 else 30.0)))
        win_frames = max(1, min(win_frames, T))

        max_start = T - win_frames
        start = random.randint(0, max_start) if (self.split == "train" and max_start > 0) else max_start // 2
        return full_lm[:, start:start + win_frames, :].clone()

    def _get_frame_indices(self, total_frames: int, fps: float) -> np.ndarray:
        """Tính toán các chỉ số frame cần lấy dựa trên miền thời gian thực và FPS riêng của video."""
        if total_frames <= 0:
            return np.array([0], dtype=int)

        fps = fps if fps > 0 else 30.0
        duration_sec = total_frames / fps

        # TRƯỜNG HỢP 1: time_interval == 0 -> Đọc frame tuần tự thông thường từ video (không bỏ cách frame)
        if self.time_interval == 0:
            if self.window_sec is None:
                return np.arange(total_frames, dtype=int)
            else:
                win_sec = min(self.window_sec, duration_sec)
                num_frames = int(round(win_sec * fps))
                num_frames = max(1, min(num_frames, total_frames))
                max_start = max(0, total_frames - num_frames)
                start_idx = random.randint(0, max_start) if (self.split == "train" and max_start > 0) else max_start // 2
                return np.arange(start_idx, start_idx + num_frames, dtype=int)

        # TRƯỜNG HỢP 2: time_interval > 0 -> Lấy mẫu cách đều theo thời gian thực (time-domain sampling)
        if self.window_sec is None:
            timestamps = np.arange(0, duration_sec, self.time_interval)
            if len(timestamps) == 0:
                timestamps = np.array([0.0])
        else:
            win = min(self.window_sec, duration_sec)
            max_start = max(0.0, duration_sec - win)
            start_sec = random.uniform(0.0, max_start) if (self.split == "train" and max_start > 0) else max_start / 2.0
            timestamps = start_sec + np.arange(0, win, self.time_interval)
            if len(timestamps) == 0:
                timestamps = np.array([start_sec])

        indices = np.rint(timestamps * fps).astype(int)
        indices = np.clip(indices, 0, total_frames - 1)
        return indices

    def _read_frames(self, path: Path, indices: np.ndarray) -> List[np.ndarray]:
        """Đọc nhanh các frame tại các chỉ số được chỉ định bằng cv2.VideoCapture."""
        cap = cv2.VideoCapture(str(path))
        if not cap.isOpened():
            raise IOError(f"Không thể mở video: {path}")

        frames = []
        curr_frame = indices[0]
        cap.set(cv2.CAP_PROP_POS_FRAMES, curr_frame)

        for target_idx in indices:
            if target_idx < curr_frame and frames:
                frames.append(frames[-1].copy())
                continue
            while curr_frame < target_idx:
                if not cap.grab():
                    break
                curr_frame += 1
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(frame)
            curr_frame += 1

        cap.release()

        if len(frames) < len(indices):
            last = frames[-1] if frames else np.zeros((480, 640, 3), dtype=np.uint8)
            frames.extend([last.copy()] * (len(indices) - len(frames)))

        return frames

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, ...]:
        sample = self.samples[index]

        # Đường nhanh: lấy chuỗi landmark từ cache RAM và cắt cửa sổ thời gian
        if self.cached is not None and not self.return_frames and sample["key"] in self.cached:
            item = self.cached[sample["key"]]
            c_fps = float(item.get("fps", sample.get("fps", 30.0)))
            c_dur = float(item.get("duration_sec", sample.get("duration_sec", 0.0)))
            landmarks_tensor = self._slice_temporal_window(item["landmarks"], c_fps, c_dur)
            label_tensor = torch.tensor(sample["label"], dtype=torch.long)
            meta = {
                "video_id": sample["video_id"],
                "path": str(sample["path"]),
                "fps": c_fps,
                "num_sampled_frames": landmarks_tensor.shape[1],
                "duration_sec": c_dur,
                "time_interval": self.cache_time_interval if self.cache_time_interval is not None else self.time_interval,
            }
            return landmarks_tensor, label_tensor, meta

        fps = sample["fps"]
        total_frames = sample["total_frames"]

        if fps <= 0 or total_frames <= 0:
            cap = cv2.VideoCapture(str(sample["path"]))
            fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            sample["fps"] = fps
            sample["total_frames"] = total_frames

        indices = self._get_frame_indices(total_frames, fps)
        frames = self._read_frames(sample["path"], indices)

        target_w, target_h = self.letterbox_size
        if target_w > 0 and target_h > 0:
            frames = [letterbox(f, new_size=(target_w, target_h)) for f in frames]

        lm68 = self.extractor.extract(frames)
        landmarks_tensor = torch.from_numpy(lm68).permute(2, 0, 1).contiguous()

        label_tensor = torch.tensor(sample["label"], dtype=torch.long)
        meta = {
            "video_id": sample["video_id"],
            "path": str(sample["path"]),
            "fps": fps,
            "num_sampled_frames": len(indices),
            "duration_sec": total_frames / fps if fps > 0 else 0.0,
            "time_interval": self.time_interval,
        }

        if self.return_frames:
            return landmarks_tensor, label_tensor, meta, frames
        return landmarks_tensor, label_tensor, meta


# Alias hỗ trợ nếu người dùng gọi FaceLandmarkDataset
FaceLandmarkDataset = GCNDataset
FaceLandmarkExtractorDataset = GCNDataset


# =====================================================================
# 4. HÀM VẼ LANDMARK VÀ HIỂN THỊ VIDEO
# =====================================================================
def get_landmark_region_color(idx: int) -> Tuple[int, int, int]:
    """Trả về màu sắc đại diện (BGR) theo từng vùng giải phẫu của hệ 68 điểm mốc."""
    if 0 <= idx <= 16:
        return (200, 200, 200)  # Viền hàm
    elif 17 <= idx <= 26:
        return (255, 190, 0)    # Lông mày trái & phải
    elif 27 <= idx <= 35:
        return (0, 255, 255)    # Sống mũi & cánh mũi
    elif 36 <= idx <= 47:
        return (0, 255, 0)      # Mắt trái & phải
    elif 48 <= idx <= 59:
        return (0, 0, 255)      # Môi ngoài
    elif 60 <= idx <= 67:
        return (0, 140, 255)    # Môi trong
    return (0, 255, 0)


def draw_landmarks_on_frame(
        frame: np.ndarray,
        landmarks_xy: np.ndarray,
        show_ids: bool | List[int] | str = False,
        font_scale: float = 0.28,
        id_color: Optional[Tuple[int, int, int] | str] = (0, 255, 255),
        point_radius: int = 2,
        point_color: Optional[Tuple[int, int, int]] = (0, 255, 0),
        draw_contours: bool = True,
) -> np.ndarray:
    """
    Vẽ 68 điểm mốc khuôn mặt và các đường viền giải phẫu lên frame ảnh,
    đồng thời hỗ trợ hiển thị ID của các điểm theo hệ 68 điểm mốc của dự án (0-67).
    """
    vis = frame.copy()
    h, w = vis.shape[:2]

    num_pts = min(68, len(landmarks_xy))
    pts = []
    for i in range(num_pts):
        x = int(landmarks_xy[i, 0] * w)
        y = int(landmarks_xy[i, 1] * h)
        pts.append((x, y))

    if draw_contours and len(pts) >= 68:
        def draw_chain(indices, closed=False, color=(0, 255, 255), thickness=1):
            for i in range(len(indices) - 1):
                cv2.line(vis, pts[indices[i]], pts[indices[i + 1]], color, thickness)
            if closed:
                cv2.line(vis, pts[indices[-1]], pts[indices[0]], color, thickness)

        draw_chain(list(range(0, 17)), closed=False, color=(200, 200, 200), thickness=1)  # Viền hàm
        draw_chain(list(range(17, 22)), closed=False, color=(255, 190, 0), thickness=1)   # Lông mày trái
        draw_chain(list(range(22, 27)), closed=False, color=(255, 190, 0), thickness=1)   # Lông mày phải
        draw_chain(list(range(27, 31)), closed=False, color=(0, 255, 255), thickness=1)   # Sống mũi
        draw_chain(list(range(31, 36)), closed=False, color=(0, 255, 255), thickness=1)   # Cánh mũi
        draw_chain(list(range(36, 42)), closed=True, color=(0, 255, 0), thickness=1)      # Mắt trái
        draw_chain(list(range(42, 48)), closed=True, color=(0, 255, 0), thickness=1)      # Mắt phải
        draw_chain(list(range(48, 60)), closed=True, color=(0, 0, 255), thickness=1)      # Môi ngoài
        draw_chain(list(range(60, 68)), closed=True, color=(0, 140, 255), thickness=1)    # Môi trong

    for i, (x, y) in enumerate(pts):
        c = point_color if point_color is not None else get_landmark_region_color(i)
        cv2.circle(vis, (x, y), point_radius, c, -1)

    if show_ids:
        if isinstance(show_ids, bool):
            target_ids = list(range(len(pts)))
        elif isinstance(show_ids, str):
            kw = show_ids.lower().strip()
            regions = {
                "jaw": list(range(0, 17)),
                "eyebrows": list(range(17, 27)),
                "left_eyebrow": list(range(17, 22)),
                "right_eyebrow": list(range(22, 27)),
                "nose": list(range(27, 36)),
                "nose_bridge": list(range(27, 31)),
                "lower_nose": list(range(31, 36)),
                "eyes": list(range(36, 48)),
                "left_eye": list(range(36, 42)),
                "right_eye": list(range(42, 48)),
                "mouth": list(range(48, 68)),
                "outer_lip": list(range(48, 60)),
                "inner_lip": list(range(60, 68)),
                "all": list(range(len(pts))),
            }
            target_ids = regions.get(kw, list(range(len(pts))))
        elif hasattr(show_ids, "__iter__"):
            target_ids = list(show_ids)
        else:
            target_ids = []

        for i in target_ids:
            if 0 <= i < len(pts):
                x, y = pts[i]
                tx = max(0, min(w - 18, x + 2))
                ty = max(10, min(h - 2, y - 2))
                text = str(i)

                if id_color is None or id_color == "region":
                    text_color = get_landmark_region_color(i)
                elif isinstance(id_color, (tuple, list)):
                    text_color = tuple(id_color)
                else:
                    text_color = (0, 255, 255)

                cv2.putText(vis, text, (tx, ty), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), 2, cv2.LINE_AA)
                cv2.putText(vis, text, (tx, ty), cv2.FONT_HERSHEY_SIMPLEX, font_scale, text_color, 1, cv2.LINE_AA)

    return vis


def visualize_frame_landmarks(
        frame: np.ndarray,
        landmarks_xy: np.ndarray,
        window_name: str = "Driver Guardian - 68 Facial Landmarks",
        show_ids: bool | List[int] | str = True,
        font_scale: float = 0.28,
        id_color: Optional[Tuple[int, int, int] | str] = (0, 255, 255),
        wait_key: bool = True,
) -> np.ndarray:
    """Hiển thị 1 frame ảnh kèm 68 điểm mốc và ID của chúng theo hệ 68 điểm của dự án."""
    vis = draw_landmarks_on_frame(
        frame=frame,
        landmarks_xy=landmarks_xy,
        show_ids=show_ids,
        font_scale=font_scale,
        id_color=id_color,
    )
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    h, w = vis.shape[:2]
    cv2.resizeWindow(window_name, min(w, 800), min(h, 600))
    cv2.imshow(window_name, vis)
    if wait_key:
        print(f"\n[Hiển thị Frame Landmark] Đang mở cửa sổ '{window_name}' (show_ids={show_ids}).")
        print(">>> Nhấn phím bất kỳ hoặc 'q' trên cửa sổ ảnh để đóng <<<")
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    return vis


def visualize_video_landmarks(
        landmarks_tensor: torch.Tensor | np.ndarray,
        frames: List[np.ndarray],
        meta: Dict[str, Any],
        label: torch.Tensor | int = 0,
        time_interval: float = 0.2,
        window_name: str = "Driver Guardian - 68 Facial Landmarks",
        show_ids: bool | List[int] | str = False,
        font_scale: float = 0.28,
        id_color: Optional[Tuple[int, int, int] | str] = (0, 255, 255),
):
    """Hiển thị video tuần tự từng frame kèm các điểm mốc landmark đã trích xuất."""
    if isinstance(landmarks_tensor, torch.Tensor):
        if landmarks_tensor.ndim == 4:
            landmarks_tensor = landmarks_tensor[0]
        if landmarks_tensor.shape[0] == 2:
            lm_np = landmarks_tensor.permute(1, 2, 0).cpu().numpy()
        else:
            lm_np = landmarks_tensor.cpu().numpy()
    elif isinstance(landmarks_tensor, np.ndarray):
        if landmarks_tensor.ndim == 3 and landmarks_tensor.shape[0] == 2:
            lm_np = np.transpose(landmarks_tensor, (1, 2, 0))
        else:
            lm_np = landmarks_tensor
    else:
        raise TypeError(f"landmarks_tensor không đúng kiểu: {type(landmarks_tensor)}")

    T = min(lm_np.shape[0], len(frames))
    fps = meta.get("fps", 30.0) or 30.0
    if time_interval == 0:
        delay_ms = max(1, int(1000 / fps))
        total_time_str = f"~{T / fps:.1f}s"
    else:
        delay_ms = max(1, int(time_interval * 1000))
        total_time_str = f"~{T * time_interval:.1f}s"

    lbl_val = label.item() if isinstance(label, torch.Tensor) else int(label)
    label_text = "ALERT (0)" if lbl_val == 0 else "DROWSY (1)"
    label_color = (0, 255, 0) if lbl_val == 0 else (0, 0, 255)

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    h, w = frames[0].shape[:2]
    cv2.resizeWindow(window_name, min(w, 800), min(h, 600))

    video_id = meta.get("video_id", "Sample")
    print(f"\n[Playing Video] {video_id} ({T} frames, {total_time_str})")
    print(">>> Phím nóng: [i] Bật/tắt ID (0-67) | [Space] Tạm dừng/Soi chi tiết | [a/d] Frame lùi/tiến | [q/ESC] Thoát <<<")

    t = 0
    paused = False
    show_ids_state = show_ids

    while t < T:
        vis = draw_landmarks_on_frame(
            frame=frames[t],
            landmarks_xy=lm_np[t],
            show_ids=show_ids_state,
            font_scale=font_scale,
            id_color=id_color,
        )

        cv2.putText(vis, f"Video: {video_id} | Status: {label_text}", (15, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 0), 3, cv2.LINE_AA)
        cv2.putText(vis, f"Video: {video_id} | Status: {label_text}", (15, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, label_color, 2, cv2.LINE_AA)

        curr_time_sec = (t / fps) if time_interval == 0 else (t * time_interval)
        time_info = f"Frame: {t + 1}/{T} | Time: {curr_time_sec:.1f}s | FPS: {fps:.1f}"
        if paused:
            time_info += "  [PAUSED - Soi ID]"
        cv2.putText(vis, time_info, (15, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 3, cv2.LINE_AA)
        cv2.putText(vis, time_info, (15, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

        id_str = "ON" if show_ids_state else "OFF"
        guide = f"[i]: IDs ({id_str}) | [Space]: {'Resume' if paused else 'Pause'} | [q/ESC]: Exit"
        cv2.putText(vis, guide, (15, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 3, cv2.LINE_AA)
        cv2.putText(vis, guide, (15, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1, cv2.LINE_AA)

        cv2.imshow(window_name, vis)

        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            print("Cửa sổ đã được đóng.")
            break

        wait_ms = 0 if paused else delay_ms
        key = cv2.waitKey(wait_ms) & 0xFF

        if key in (ord('q'), ord('Q'), 27):
            print("Đã nhận lệnh dừng từ người dùng.")
            break
        elif key in (ord('i'), ord('I')):
            show_ids_state = not show_ids_state
            print(f">>> Landmark IDs (0-67): {'BẬT' if show_ids_state else 'TẮT'}")
            continue
        elif key == 32:
            paused = not paused
            print(f">>> {'TẠM DỪNG' if paused else 'TIẾP TỤC'}")
            continue
        elif paused and key in (ord('d'), ord('D'), 83):
            t = min(T - 1, t + 1)
            continue
        elif paused and key in (ord('a'), ord('A'), 81):
            t = max(0, t - 1)
            continue

        if not paused:
            t += 1

    cv2.destroyAllWindows()


# =====================================================================
# 5. HÀM COLLATE VÀ BUILD DATALOADER
# =====================================================================
def collate_gcn(batch: List[Tuple[Any, ...]]):
    """
    Gom các mẫu vào batch.
    Khi window_sec=None (số frame T giữa các video có thể khác nhau) hoặc các video có độ dài khác nhau,
    tự động pad thêm frame 0 ở chiều thời gian T về max_T của batch và trả về tensor lengths.

    Returns:
        - padded_landmarks: Tensor (B, 2, max_T, 68)
        - labels          : LongTensor (B,)
        - lengths         : LongTensor (B,) chứa số frame thực tế trước khi pad của từng mẫu
        - metas           : List[Dict[str, Any]]
        - frames (tùy chọn): List[List[np.ndarray]] nếu dataset bật return_frames=True
    """
    has_frames = len(batch[0]) == 4
    if has_frames:
        landmarks, labels, metas, frames_list = zip(*batch)
    else:
        landmarks, labels, metas = zip(*batch)

    lengths = [lm.shape[1] for lm in landmarks]
    max_len = max(lengths)

    padded_landmarks = []
    updated_metas = []
    for lm, meta, t in zip(landmarks, metas, lengths):
        if t < max_len:
            pad_tensor = torch.zeros((lm.shape[0], max_len, lm.shape[2]), dtype=lm.dtype)
            pad_tensor[:, :t, :] = lm
            padded_landmarks.append(pad_tensor)
        else:
            padded_landmarks.append(lm)

        meta_copy = dict(meta)
        meta_copy["valid_frames"] = t
        meta_copy["padded_frames"] = max_len
        updated_metas.append(meta_copy)

    lengths_tensor = torch.tensor(lengths, dtype=torch.long)

    if has_frames:
        return torch.stack(padded_landmarks), torch.stack(labels), lengths_tensor, updated_metas, list(frames_list)
    return torch.stack(padded_landmarks), torch.stack(labels), lengths_tensor, updated_metas


def build_gcn_dataloaders(
        dataset_dir: str | Path = r"E:\LSTM\data_processed",
        batch_size: int = 16,
        num_workers: int = 0,
        time_interval: float = 0.1,
        window_sec: Optional[float] = 10.0,
        manifest_filename: str = "dataset_merged_split.csv",
        pin_memory: bool = True,
        shuffle_train: bool = True,
        landmark_model_path: Optional[str | Path] = None,
        letterbox_size: int | Tuple[int, int] = (640, 480),
        device: str = "cuda",
        det_size: Tuple[int, int] = (320, 320),
        model_name: str = "buffalo_l",
        use_cache: bool = True,
        cache_dir: Optional[str | Path] = None,
        landmark_batch_size: int = 64,
):
    """
    Xây dựng nhanh bộ DataLoader cho cả 3 tập train, val, test.
    Ưu tiên nạp landmark từ cache .pt ({cache_dir}/{split}.pt) nếu use_cache=True.
    """
    common = dict(
        dataset_dir=dataset_dir,
        time_interval=time_interval,
        window_sec=window_sec,
        manifest_filename=manifest_filename,
        landmark_model_path=landmark_model_path,
        letterbox_size=letterbox_size,
        device=device,
        det_size=det_size,
        model_name=model_name,
        use_cache=use_cache,
        cache_dir=cache_dir,
        landmark_batch_size=landmark_batch_size,
    )
    train_ds = GCNDataset(split="train", **common)
    val_ds = GCNDataset(split="val", **common)
    test_ds = GCNDataset(split="test", **common)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=shuffle_train, num_workers=num_workers,
        pin_memory=pin_memory, collate_fn=collate_gcn
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers,
        pin_memory=pin_memory, collate_fn=collate_gcn
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers,
        pin_memory=pin_memory, collate_fn=collate_gcn
    )
    return train_loader, val_loader, test_loader

In [7]:
# Cell 5: Các khối xây dựng Đồ thị Không - Thời gian và Đầu ra Chú ý (block.py)
class FacialGraphBuilder:
    """
    Xây dựng cấu trúc đồ thị 68 điểm mốc khuôn mặt bằng torch_geometric.utils.
    Hỗ trợ xuất cả edge_index (Sparse) và Adjacency Matrix (Dense).
    """

    @staticmethod
    def get_edge_index(
            num_nodes: int = 68,
            vertical_pairs: Optional[List[Tuple[int, int]] | List[List[int]]] = None,
            cross_region_pairs: Optional[List[Tuple[int, int]] | List[List[int]]] = None,
    ) -> torch.Tensor:
        edges = []

        def add_chain(indices, closed=False):
            for i in range(len(indices) - 1):
                edges.append([indices[i], indices[i + 1]])
            if closed:
                edges.append([indices[-1], indices[0]])

        # 1. Cạnh giải phẫu cơ mặt (Anatomical Contours)
        add_chain(range(0, 17), closed=False)  # Viền hàm
        add_chain(range(17, 22), closed=False)  # Lông mày trái
        add_chain(range(22, 27), closed=False)  # Lông mày phải
        add_chain(range(27, 31), closed=False)  # Sống mũi
        add_chain(range(31, 36), closed=False)  # Cánh mũi
        add_chain(range(36, 42), closed=True)  # Mắt trái
        add_chain(range(42, 48), closed=True)  # Mắt phải
        add_chain(range(48, 60), closed=True)  # Môi ngoài
        add_chain(range(60, 68), closed=True)  # Môi trong

        # 2. Cạnh dọc đo độ mở Mắt (EAR) & Miệng (MAR)
        if vertical_pairs is None:
            vertical_pairs = [
                [37, 41], [38, 40], [43, 47], [44, 46],  # Mí mắt trên - dưới
                [50, 58], [51, 57], [52, 56], [61, 67], [62, 66], [63, 65]  # Môi trên - dưới
            ]

        # 3. Cạnh tiên nghiệm liên vùng (Mắt <-> Miệng, Mũi <-> Cằm)
        if cross_region_pairs is None:
            cross_region_pairs = [
                [37, 48], [41, 48], [38, 62], [40, 66],  # Mắt trái <-> Miệng
                [44, 54], [46, 54], [43, 62], [47, 66],  # Mắt phải <-> Miệng
                [30, 8], [30, 36], [30, 45], [27, 8]  # Chóp mũi <-> Cằm & Đuôi mắt
            ]

        all_edges = edges + [list(p) for p in vertical_pairs] + [list(p) for p in cross_region_pairs]
        edge_index = torch.tensor(all_edges, dtype=torch.long).t().contiguous()

        # Sử dụng hàm có sẵn của PyTorch Geometric để tạo đồ thị vô hướng và thêm self-loops
        edge_index = to_undirected(edge_index, num_nodes=num_nodes)
        edge_index, _ = add_self_loops(edge_index, num_nodes=num_nodes)
        return edge_index

    @classmethod
    def get_dense_adj(
            cls,
            num_nodes: int = 68,
            vertical_pairs: Optional[List[Tuple[int, int]] | List[List[int]]] = None,
            cross_region_pairs: Optional[List[Tuple[int, int]] | List[List[int]]] = None,
    ) -> torch.Tensor:
        """Trả về ma trận kề Dense kích thước (V, V) từ edge_index."""
        edge_index = cls.get_edge_index(
            num_nodes=num_nodes,
            vertical_pairs=vertical_pairs,
            cross_region_pairs=cross_region_pairs,
        )
        return to_dense_adj(edge_index, max_num_nodes=num_nodes).squeeze(0)


class KinematicFeatureExtractor(nn.Module):
    """
    Khối tiền xử lý chuẩn hóa tọa độ 68 điểm mốc:
    1. Đưa tọa độ tĩnh (x, y) về Hệ tọa độ Cục bộ Tâm mũi (điểm 30) & chia cho khoảng cách 2 mắt (36 <-> 45).
    2. Tính Vận tốc (dx, dy) và Gia tốc (d2x, d2y) đã chuẩn hóa theo kích thước khuôn mặt
       (giữ nguyên được quỹ đạo rơi của đầu khi tài xế ngủ gật).
    Hỗ trợ temporal mask để triệt tiêu cú sốc số học tại biên zero-padding khi batch có độ dài thay đổi.
    """

    def __init__(
            self,
            nose_idx: int = 30,
            left_eye_idx: int = 36,
            right_eye_idx: int = 45,
            eps: float = 1e-6,
    ):
        super().__init__()
        self.nose_idx = nose_idx
        self.left_eye_idx = left_eye_idx
        self.right_eye_idx = right_eye_idx
        self.eps = eps

    @staticmethod
    def build_temporal_mask(
            x: torch.Tensor,
            lengths: Optional[torch.Tensor] = None,
            eps: float = 1e-6,
    ) -> torch.Tensor:
        """
        Tạo binary float mask kích thước (B, 1, T, 1) biểu diễn các frame hợp lệ (1.0) và frame đệm (0.0).
        - Nếu lengths được truyền vào: tạo mask chính xác từ lengths.
        - Nếu lengths is None: tự động nhận diện từ biên độ tọa độ (frame hợp lệ có tọa độ khác 0).
        """
        B, _, T, _ = x.shape
        device = x.device
        if lengths is not None:
            t_idx = torch.arange(T, device=device).unsqueeze(0)  # (1, T)
            mask = (t_idx < lengths.to(device).unsqueeze(1)).float()  # (B, T)
            return mask.view(B, 1, T, 1)
        else:
            # Tự nhận diện frame hợp lệ (tổng tọa độ x, y khác 0)
            frame_activity = x[:, :2].abs().sum(dim=(1, 3), keepdim=True)  # (B, 1, T, 1)
            return (frame_activity > eps).float()

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Input : (B, 2, T, V=68) - Tọa độ pixel hoặc [0, 1] trong Hệ tọa độ Khung ảnh.
        mask  : (B, 1, T, 1)    - Float mask (1.0 = valid, 0.0 = padded)
        Output: (B, 6, T, V=68) - [x_norm, y_norm, dx_norm, dy_norm, d2x_norm, d2y_norm]
        """
        if x.size(1) != 2:
            return x * mask if mask is not None else x

        # 1. Tính khoảng cách giải phẫu chuẩn d_t (Inter-Ocular Distance giữa điểm 36 và 45)
        # eye_dist shape: (B, 1, T, 1)
        p_left_eye = x[:, :, :, self.left_eye_idx: self.left_eye_idx + 1]  # (B, 2, T, 1)
        p_right_eye = x[:, :, :, self.right_eye_idx: self.right_eye_idx + 1]  # (B, 2, T, 1)
        raw_iod = torch.norm(p_right_eye - p_left_eye, p=2, dim=1, keepdim=True)
        # Tránh chia cho eps=1e-6 tại các frame pad gây spike số học khổng lồ
        iod_scale = raw_iod.clamp(min=self.eps)

        # 2. Chuẩn hóa tọa độ hình thái tĩnh theo Tâm mũi (điểm 30) và tỷ lệ IOD
        nose_pos = x[:, :, :, self.nose_idx: self.nose_idx + 1]  # (B, 2, T, 1)
        pos_norm = (x - nose_pos) / iod_scale  # (B, 2, T, V)

        # 3. Tính Vận tốc (dx, dy) và Gia tốc (d2x, d2y) chuẩn hóa theo IOD
        # (Tính diff trên x trước khi trừ mũi để giữ được vận tốc gật đầu của cả khuôn mặt)
        T = x.size(2)
        if T <= 1:
            vel_norm = torch.zeros_like(pos_norm)
            acc_norm = torch.zeros_like(pos_norm)
        else:
            vel_raw = torch.diff(x, n=1, dim=2)
            if mask is not None:
                # Triệt tiêu bước nhảy biên giữa frame hợp lệ cuối cùng và frame pad đầu tiên TRƯỚC khi replicate pad
                pair_valid = mask[:, :, 1:, :] * mask[:, :, :-1, :]
                vel_raw = vel_raw * pair_valid
            vel_raw = F.pad(vel_raw, (0, 0, 1, 0), mode='replicate')

            vel_norm = vel_raw / iod_scale  # (B, 2, T, V)
            if mask is not None:
                vel_norm = vel_norm * mask

            acc_raw = torch.diff(vel_norm, n=1, dim=2)
            if mask is not None:
                acc_raw = acc_raw * pair_valid
            acc_norm = F.pad(acc_raw, (0, 0, 1, 0), mode='replicate')  # (B, 2, T, V)

        # Ghép thành 6 kênh đặc trưng chuẩn hóa hoàn chỉnh
        out = torch.cat([pos_norm, vel_norm, acc_norm], dim=1)  # (B, 6, T, V)

        if mask is not None:
            out = out * mask

        return out


class PyGSpatialGraphConv(nn.Module):
    """
    Khối tích chập không gian tận dụng DenseGCNConv và DenseGATConv của PyTorch Geometric.
    - DenseGCNConv: Học trên ma trận tiên nghiệm A_prior + ma trận tham số tự học A_adaptive.
    - DenseGATConv: Tự động tính Attention động (A_dynamic) giữa các điểm mốc theo từng mẫu.
    """

    def __init__(
            self,
            in_channels: int,
            out_channels: int,
            adj_prior: torch.Tensor,
            gat_heads: int = 4,
            dropout: float = 0.1,
            adaptive_init_scale: float = 1e-5,
    ):
        super().__init__()
        num_nodes = adj_prior.size(0)
        self.register_buffer('adj_prior', adj_prior)

        # Ma trận kề tự học toàn cục A_adaptive
        self.adj_adaptive = nn.Parameter(torch.randn(num_nodes, num_nodes) * adaptive_init_scale)

        # 1. Nhánh GCN tĩnh + thích ứng (PyG DenseGCNConv tự động chuẩn hóa D^-1/2 A D^-1/2)
        self.gcn = DenseGCNConv(in_channels, out_channels, improved=False, bias=True)

        # 2. Nhánh Graph Attention động (PyG DenseGATConv)
        self.gat = DenseGATConv(
            in_channels,
            out_channels,
            heads=gat_heads,
            concat=False,
            dropout=dropout,
            bias=True
        )
        self.alpha = nn.Parameter(torch.zeros(1))  # Hệ số hòa trộn nhánh GAT động

        self.bn = nn.BatchNorm2d(out_channels)
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x shape: (B, C_in, T, V) -> Output shape: (B, C_out, T, V)
        """
        B, C, T, V = x.shape

        # Chuyển chiều về dạng chuẩn của PyG Dense layers: (B * T, V, C_in)
        x_dense = x.permute(0, 2, 3, 1).contiguous().view(B * T, V, C)

        # Tổng hợp ma trận kề cho nhánh GCN
        adj_total = self.adj_prior + self.adj_adaptive

        # Thực thi DenseGCNConv và DenseGATConv (add_loop=False vì adj_prior đã có self-loops)
        out_gcn = self.gcn(x_dense, adj_total, add_loop=False)  # (B*T, V, C_out)
        out_gat = self.gat(x_dense, self.adj_prior, add_loop=False)  # (B*T, V, C_out)

        out = out_gcn + self.alpha * out_gat  # (B*T, V, C_out)

        # Khôi phục lại chiều (B, C_out, T, V)
        out = out.view(B, T, V, -1).permute(0, 3, 1, 2).contiguous()
        return self.act(self.bn(out))


class MultiScaleTemporalConv(nn.Module):
    """
    Khối Tích chập Thời gian Đa tỷ lệ (Multi-Scale TCN) sử dụng nn.Conv2d với Dilation khác nhau:
    - Nhánh 1 (Dilation = 1): Bắt chuyển động nhanh (chớp mắt).
    - Nhánh 2 (Dilation = 2): Bắt chuyển động kéo dài (ngáp, ngủ gật).
    """

    def __init__(
            self,
            channels: int,
            kernel_size: int = 5,
            stride: int = 1,
            dropout: float = 0.2,
            dilations: Tuple[int, int] = (1, 2),
    ):
        super().__init__()
        mid_channels = channels // 2
        d1, d2 = dilations
        pad_d1 = (kernel_size - 1) * d1 // 2
        pad_d2 = (kernel_size - 1) * d2 // 2

        self.branch1 = nn.Conv2d(
            channels, mid_channels,
            kernel_size=(kernel_size, 1),
            stride=(stride, 1),
            padding=(pad_d1, 0),
            dilation=(d1, 1)
        )
        self.branch2 = nn.Conv2d(
            channels, channels - mid_channels,
            kernel_size=(kernel_size, 1),
            stride=(stride, 1),
            padding=(pad_d2, 0),
            dilation=(d2, 1)
        )

        self.bn = nn.BatchNorm2d(channels)
        self.drop = nn.Dropout(dropout, inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = torch.cat([self.branch1(x), self.branch2(x)], dim=1)
        return self.drop(self.bn(out))


class STGCNBlock(nn.Module):
    """
    Khối ST-GCN hoàn chỉnh: PyGSpatialGraphConv -> MultiScaleTemporalConv -> Residual Connection.
    Hỗ trợ temporal mask để triệt tiêu bias rò rỉ tại các frame zero-padding.
    """

    def __init__(
            self,
            in_channels: int,
            out_channels: int,
            adj_prior: torch.Tensor,
            temporal_kernel: int = 5,
            stride: int = 1,
            dropout: float = 0.2,
            gat_heads: int = 4,
            adaptive_init_scale: float = 1e-5,
            dilations: Tuple[int, int] = (1, 2),
    ):
        super().__init__()
        self.stride = stride
        self.sgcn = PyGSpatialGraphConv(
            in_channels,
            out_channels,
            adj_prior,
            gat_heads=gat_heads,
            dropout=dropout,
            adaptive_init_scale=adaptive_init_scale,
        )
        self.tgcn = MultiScaleTemporalConv(
            out_channels,
            kernel_size=temporal_kernel,
            stride=stride,
            dropout=dropout,
            dilations=dilations,
        )

        if in_channels == out_channels and stride == 1:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(stride, 1), bias=False),
                nn.BatchNorm2d(out_channels)
            )

        self.act = nn.GELU()

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        x   : (B, C_in, T_in, V)
        mask: Optional (B, 1, T_in, 1) - Float mask (1.0 = valid, 0.0 = padded)
        """
        res = self.residual(x)
        x = self.sgcn(x)
        if mask is not None:
            # Triệt tiêu bias của GCN/GAT/BN ở vùng pad trước khi tích chập thời gian
            x = x * mask
        x = self.tgcn(x)
        out = self.act(x + res)
        if mask is not None:
            mask_out = mask[:, :, ::self.stride, :][:, :, :out.size(2), :]
            out = out * mask_out
        return out


class _GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, lambda_grl: float):
        ctx.lambda_grl = lambda_grl
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        return -ctx.lambda_grl * grad_output, None


class GradientReversalLayer(nn.Module):
    """Lớp Đảo ngược Gradient (GRL) dùng cho Domain Generalization theo bài báo MSTGCN."""

    def __init__(self, lambda_grl: float = 1.0):
        super().__init__()
        self.lambda_grl = lambda_grl

    def set_lambda(self, progress: float, gamma: float = 10.0):
        """
        Tự động tăng dần lambda_grl từ 0.0 -> 1.0 theo tiến độ huấn luyện progress thuộc [0, 1]
        giúp mạng học ổn định ở những epoch đầu tiên.
        """
        self.lambda_grl = float(2.0 / (1.0 + math.exp(-gamma * progress)) - 1.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return _GradientReversalFunction.apply(x, self.lambda_grl)


class AttentiveDrowsinessOutputHead(nn.Module):
    """
    Khối Dự đoán Đầu ra Tối ưu cho ST-GCN Phát hiện Buồn ngủ:
    1. Spatial Node Attention: Tự động tập trung vào các điểm mốc Mắt/Miệng, loại bỏ nhiễu viền hàm.
    2. Temporal Gated-Conv + Max & Attention Pooling: Kế thừa tầng Output Temporal Conv của Yu et al.
       kết hợp Max-Pooling để bắt khoảnh khắc chợp mắt/ngáp ngắn (Micro-sleep) trong chuỗi dài.
    3. Dual Heads (G_y & G_d): Hỗ trợ đồng thời Phân loại buồn ngủ (G_y), Phân loại miền đối kháng (G_d - Jia et al.)
       và Hồi quy mức độ mệt mỏi liên tục (KSS Score Regression).
    """

    def __init__(
            self,
            in_channels: int = 256,
            num_classes: int = 2,
            num_nodes: int = 68,
            num_domains: int = 0,  # Đặt > 0 nếu muốn bật nhánh Domain Classifier (G_d) của MSTGCN
            enable_regression: bool = False,  # Đặt True nếu muốn dự đoán thêm điểm buồn ngủ liên tục (KSS)
            dropout: float = 0.25,
            classifier_hidden_dim: Optional[int] = None,
    ):
        super().__init__()
        self.num_nodes = num_nodes
        self.num_domains = num_domains
        self.enable_regression = enable_regression
        spatial_mid = max(1, in_channels // 4)
        hidden_dim = classifier_hidden_dim if classifier_hidden_dim is not None else max(1, in_channels // 2)

        # --- BƯỚC 1: GOM ĐẶC TRƯNG KHÔNG GIAN CÓ CHÚ Ý (Spatial Node Attention) ---
        # Học trọng số quan trọng cho từng điểm trong 68 điểm mốc khuôn mặt
        self.spatial_attn = nn.Sequential(
            nn.Conv2d(in_channels, spatial_mid, kernel_size=1),
            nn.GELU(),
            nn.Conv2d(spatial_mid, 1, kernel_size=1),
            nn.Softmax(dim=-1)  # Chuẩn hóa trọng số dọc theo 68 đỉnh (dim V)
        )

        # --- BƯỚC 2: TINH CHỈNH & GOM ĐẶC TRƯNG THỜI GIAN (Theo Yu et al. + Attention) ---
        # Lớp Temporal Conv cuối cùng kèm cổng GLU lọc nhiễu chuỗi thời gian
        self.final_temporal_conv = nn.Sequential(
            nn.Conv1d(in_channels, in_channels * 2, kernel_size=3, padding=1),
            nn.GLU(dim=1),
            nn.BatchNorm1d(in_channels)
        )
        # Trọng số chú ý thời gian (Temporal Attention) bắt khung hình buồn ngủ
        self.temporal_attn = nn.Sequential(
            nn.Conv1d(in_channels, 1, kernel_size=1),
            nn.Softmax(dim=-1)  # Chuẩn hóa dọc theo trục thời gian T
        )

        # Kết hợp cả Temporal Attention Pooling và Temporal Max Pooling -> Nhân đôi chiều đặc trưng
        fused_dim = in_channels * 2

        # --- BƯỚC 3: CÁC NHÁNH DỰ ĐOÁN ĐẦU RA (Prediction Heads) ---
        # 3.1. Nhánh chính G_y: Phân loại trạng thái buồn ngủ (Công thức 11 - MSTGCN)
        self.label_predictor = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Dropout(dropout),
            nn.Linear(fused_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim, num_classes)
        )

        # 3.2. Nhánh phụ G_d (Tùy chọn): Phân loại miền/tài xế qua GRL (Công thức 12 - MSTGCN)
        if num_domains > 0:
            self.grl = GradientReversalLayer(lambda_grl=1.0)
            self.domain_classifier = nn.Sequential(
                nn.LayerNorm(fused_dim),
                nn.Dropout(dropout),
                nn.Linear(fused_dim, hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, num_domains)
            )

        # 3.3. Nhánh Hồi quy (Tùy chọn): Dự đoán chỉ số buồn ngủ liên tục (như KSS / PERCLOS)
        if enable_regression:
            self.regression_head = nn.Sequential(
                nn.LayerNorm(fused_dim),
                nn.Linear(fused_dim, 1),
                nn.Sigmoid()  # Quy về khoảng [0.0 -> 1.0] (Tỉnh táo -> Ngủ gật hoàn toàn)
            )

    def pool_spatio_temporal(
            self,
            feat: torch.Tensor,
            mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Nén Tensor (B, C, T, V=68) -> Vector đặc trưng toàn cục (B, 2*C)
        mà không làm mất tín hiệu cục bộ ở Mắt/Miệng hay các cơn chợp mắt ngắn.
        Hỗ trợ mask thời gian (B, 1, T, 1) hoặc (B, 1, T) để loại bỏ hoàn toàn frame pad.
        """
        t_mask: Optional[torch.Tensor] = None
        if mask is not None:
            if mask.dim() == 4:
                t_mask = mask.squeeze(-1)  # (B, 1, T)
            elif mask.dim() == 3:
                t_mask = mask  # (B, 1, T)
            else:
                t_mask = mask.view(feat.size(0), 1, feat.size(2))
            t_mask = t_mask[:, :, :feat.size(2)].to(dtype=feat.dtype)

        # 1. Gộp chiều Không gian (V=68 -> 1) bằng Spatial Attention
        s_weights = self.spatial_attn(feat)  # (B, 1, T, V)
        x_spatial = torch.sum(feat * s_weights, dim=-1)  # (B, C, T)
        if t_mask is not None:
            x_spatial = x_spatial * t_mask

        # 2. Lọc qua lớp Temporal Conv cuối cùng (Yu et al.)
        x_temp = self.final_temporal_conv(x_spatial) + x_spatial  # (B, C, T)
        if t_mask is not None:
            x_temp = x_temp * t_mask

        # 3. Gộp chiều Thời gian (T -> 1): Kết hợp Attention Pool (xu hướng chung) + Max Pool (đỉnh cơn ngáp/nhắm mắt)
        if t_mask is not None:
            is_pad = (t_mask <= 0.5)  # (B, 1, T) bool
            # Trường hợp biên: nếu có mẫu toàn bộ frame đều bị pad (length=0), mở lại frame 0 để tránh NaN/Inf
            all_pad = is_pad.all(dim=-1, keepdim=True)  # (B, 1, 1)
            safe_is_pad = is_pad & (~all_pad)

            attn_logits = self.temporal_attn[0](x_temp)  # (B, 1, T)
            neg_fill_attn = -1e4 if attn_logits.dtype == torch.float16 else -1e9
            attn_logits = attn_logits.masked_fill(safe_is_pad, neg_fill_attn)
            t_weights = self.temporal_attn[1](attn_logits)  # Softmax(dim=-1)
            t_weights = t_weights * t_mask
            t_attn_pool = torch.sum(x_temp * t_weights, dim=-1)  # (B, C)

            neg_fill_temp = -1e4 if x_temp.dtype == torch.float16 else -1e9
            x_temp_masked = x_temp.masked_fill(safe_is_pad, neg_fill_temp)
            t_max_pool = torch.amax(x_temp_masked, dim=-1)  # (B, C)
            t_max_pool = t_max_pool * (~all_pad.squeeze(-1)).to(dtype=t_max_pool.dtype)
        else:
            t_weights = self.temporal_attn(x_temp)  # (B, 1, T)
            t_attn_pool = torch.sum(x_temp * t_weights, dim=-1)  # (B, C)
            t_max_pool = torch.amax(x_temp, dim=-1)  # (B, C)

        return torch.cat([t_attn_pool, t_max_pool], dim=-1)  # (B, 2*C)

    def forward(
            self,
            feat: torch.Tensor,
            mask: Optional[torch.Tensor] = None,
            return_domain: bool = False,
            return_probs: bool = False,
    ):
        """
        Input :
          - feat: shape (B, C, T, V) từ khối STGCNBlock cuối cùng.
          - mask: Optional shape (B, 1, T, 1) hoặc (B, 1, T) đánh dấu các frame hợp lệ.
        Output:
          - Mặc định (Inference/Train): y_logits (hoặc xác suất Softmax nếu return_probs=True).
          - Khi huấn luyện đa tác vụ / Domain Generalization: dict chứa 'logits', 'domain_logits', 'fatigue_score'.
        """
        global_feat = self.pool_spatio_temporal(feat, mask=mask)  # (B, 2*C)

        # Dự đoán nhãn trạng thái buồn ngủ (G_y)
        y_logits = self.label_predictor(global_feat)  # (B, num_classes)
        y_out = F.softmax(y_logits, dim=-1) if return_probs else y_logits

        if not return_domain and not self.enable_regression:
            return y_out

        outputs = {"logits": y_out}

        if return_domain and self.num_domains > 0:
            rev_feat = self.grl(global_feat)
            outputs["domain_logits"] = self.domain_classifier(rev_feat)

        if self.enable_regression:
            outputs["fatigue_score"] = self.regression_head(global_feat).squeeze(-1)

        return outputs

In [8]:
# Cell 6: Kiến trúc mô hình DriverSTGCN và DriverSTGCNTransformer (model.py)
class DriverSTGCN(nn.Module):
    def __init__(
            self,
            in_channels: int = 6,
            num_classes: int = 2,
            num_nodes: int = 68,
            num_domains: int = 0,  # > 0 để bật Domain Generalization (MSTGCN)
            enable_regression: bool = False,  # True để bật dự đoán điểm buồn ngủ liên tục
            hidden_dims: Optional[List[int]] = None,
            strides: Optional[List[int]] = None,
            dropout: float = 0.25,
            temporal_kernel: int = 5,
            gat_heads: int = 4,
            classifier_hidden_dim: Optional[int] = None,
            nose_idx: int = 30,
            left_eye_idx: int = 36,
            right_eye_idx: int = 45,
            eps: float = 1e-6,
            adaptive_init_scale: float = 1e-5,
            vertical_pairs: Optional[List[Tuple[int, int]]] = None,
            cross_region_pairs: Optional[List[Tuple[int, int]]] = None,
    ):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [64, 128, 256]
        else:
            hidden_dims = list(hidden_dims)

        if strides is None:
            strides = [1, 2, 2]
        else:
            strides = list(strides)

        if len(hidden_dims) == 0 or len(hidden_dims) != len(strides):
            raise ValueError(
                f"hidden_dims ({len(hidden_dims)}) và strides ({len(strides)}) phải khác rỗng và có cùng độ dài."
            )

        # Nếu truyền in_channels=2 (tọa độ thô x, y), sau KinematicFeatureExtractor sẽ luôn có 6 kênh động học
        kinematic_channels = 6 if in_channels == 2 else in_channels

        adj_prior = FacialGraphBuilder.get_dense_adj(
            num_nodes=num_nodes,
            vertical_pairs=vertical_pairs,
            cross_region_pairs=cross_region_pairs,
        )
        self.kinematics = KinematicFeatureExtractor(
            nose_idx=nose_idx,
            left_eye_idx=left_eye_idx,
            right_eye_idx=right_eye_idx,
            eps=eps,
        )
        self.input_bn = nn.BatchNorm1d(kinematic_channels * num_nodes)

        blocks = []
        curr_dim = kinematic_channels
        for out_dim, stride in zip(hidden_dims, strides):
            blocks.append(
                STGCNBlock(
                    curr_dim,
                    out_dim,
                    adj_prior,
                    temporal_kernel=temporal_kernel,
                    stride=stride,
                    dropout=dropout,
                    gat_heads=gat_heads,
                    adaptive_init_scale=adaptive_init_scale,
                )
            )
            curr_dim = out_dim
        self.backbone = nn.ModuleList(blocks)

        # Sử dụng Khối Dự đoán Đầu ra Tối ưu mới
        self.output_head = AttentiveDrowsinessOutputHead(
            in_channels=curr_dim,
            num_classes=num_classes,
            num_nodes=num_nodes,
            num_domains=num_domains,
            enable_regression=enable_regression,
            dropout=dropout,
            classifier_hidden_dim=classifier_hidden_dim,
        )

    def forward(
            self,
            x: torch.Tensor,
            lengths: Optional[torch.Tensor] = None,
            return_domain: bool = False,
            return_probs: bool = False,
    ):
        """
        x            : (B, 2, T, V) hoặc (B, 6, T, V)
        lengths      : Optional LongTensor (B,) - số frame hợp lệ trước khi pad
        return_domain: Trả thêm domain_logits (nếu num_domains > 0)
        return_probs : Áp dụng Softmax trên logits đầu ra
        """
        mask = self.kinematics.build_temporal_mask(x, lengths=lengths)
        x = self.kinematics(x, mask=mask)
        B, C, T, V = x.shape

        x = x.permute(0, 1, 3, 2).contiguous().view(B, C * V, T)
        x = self.input_bn(x)
        x = x.view(B, C, V, T).permute(0, 1, 3, 2).contiguous()
        x = x * mask

        curr_mask = mask
        for block in self.backbone:
            x = block(x, mask=curr_mask)
            curr_mask = curr_mask[:, :, ::block.stride, :][:, :, :x.size(2), :]

        return self.output_head(
            x,
            mask=curr_mask,
            return_domain=return_domain,
            return_probs=return_probs,
        )


class DriverSTGCNTransformer(nn.Module):
    """
    Model 2: Kiến trúc Lai ST-GCN + PyTorch TransformerEncoder.
    Dùng ST-GCN để học tương quan đồ thị cơ mặt cục bộ, sau đó đưa qua
    nn.TransformerEncoder để học phụ thuộc chuỗi thời gian dài hạn.
    Khởi tạo thuần túy qua các tham số thông thường, hoàn toàn độc lập với đối tượng config.
    """

    def __init__(
            self,
            in_channels: int = 6,
            num_classes: int = 2,
            num_nodes: int = 68,
            d_model: int = 128,
            nhead: int = 4,
            num_transformer_layers: int = 2,
            max_len: int = 300,
            dropout: float = 0.2,
            stem_channels: Optional[List[int]] = None,
            stem_strides: Optional[List[int]] = None,
            dim_feedforward: Optional[int] = None,
            init_std: float = 0.02,
            nose_idx: int = 30,
            left_eye_idx: int = 36,
            right_eye_idx: int = 45,
            eps: float = 1e-6,
            adaptive_init_scale: float = 1e-5,
            vertical_pairs: Optional[List[Tuple[int, int]]] = None,
            cross_region_pairs: Optional[List[Tuple[int, int]]] = None,
    ):
        super().__init__()
        if stem_channels is None:
            stem_channels = [64, d_model]
        if stem_strides is None:
            stem_strides = [1, 2]
        if dim_feedforward is None:
            dim_feedforward = d_model * 2

        kinematic_channels = 6 if in_channels == 2 else in_channels

        adj_prior = FacialGraphBuilder.get_dense_adj(
            num_nodes=num_nodes,
            vertical_pairs=vertical_pairs,
            cross_region_pairs=cross_region_pairs,
        )

        self.kinematics = KinematicFeatureExtractor(
            nose_idx=nose_idx,
            left_eye_idx=left_eye_idx,
            right_eye_idx=right_eye_idx,
            eps=eps,
        )
        self.input_bn = nn.BatchNorm1d(kinematic_channels * num_nodes)

        # 1. Tầng ST-GCN Stem rút trích đặc trưng đồ thị khuôn mặt
        self.stgcn_stem = nn.ModuleList([
            STGCNBlock(
                kinematic_channels,
                stem_channels[0],
                adj_prior,
                stride=stem_strides[0],
                dropout=dropout,
                adaptive_init_scale=adaptive_init_scale,
            ),
            STGCNBlock(
                stem_channels[0],
                stem_channels[1],
                adj_prior,
                stride=stem_strides[1],
                dropout=dropout,
                adaptive_init_scale=adaptive_init_scale,
            )
        ])

        # 2. Trọng số chú ý không gian (Spatial Node Attention Pooling) gộp V=68 đỉnh về 1 vector/frame
        self.node_attention = nn.Sequential(
            nn.Conv1d(stem_channels[-1], 1, kernel_size=1),
            nn.Softmax(dim=-1)
        )

        # 3. Khối TransformerEncoder chuẩn của PyTorch dọc trục thời gian T
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * init_std)
        self.pos_embedding = nn.Parameter(torch.randn(1, max_len + 1, d_model) * init_std)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_transformer_layers,
            enable_nested_tensor=False,
        )

        # 4. Đầu dự đoán trạng thái buồn ngủ
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, num_classes)
        )

    def forward(
            self,
            x: torch.Tensor,
            lengths: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        x      : (B, 2, T, V) hoặc (B, 6, T, V)
        lengths: LongTensor (B,) [Tùy chọn - số frame hợp lệ trước khi pad]
        Output : (B, num_classes)
        """
        mask = self.kinematics.build_temporal_mask(x, lengths=lengths)
        x = self.kinematics(x, mask=mask)
        B, C, T, V = x.shape

        x = x.permute(0, 1, 3, 2).contiguous().view(B, C * V, T)
        x = self.input_bn(x)
        x = x.view(B, C, V, T).permute(0, 1, 3, 2).contiguous()
        x = x * mask

        # Qua ST-GCN Stem:
        curr_mask = mask
        for block in self.stgcn_stem:
            x = block(x, mask=curr_mask)
            curr_mask = curr_mask[:, :, ::block.stride, :][:, :, :x.size(2), :]

        _, D, T_prime, _ = x.shape

        # Gộp 68 đỉnh bằng Node Attention cho mỗi bước thời gian
        x_flat = x.permute(0, 2, 1, 3).contiguous().view(B * T_prime, D, V)
        attn_weights = self.node_attention(x_flat)  # (B*T', 1, V)
        x_spatial_pooled = torch.sum(x_flat * attn_weights, dim=-1)  # (B*T', D)
        seq = x_spatial_pooled.view(B, T_prime, D)  # (B, T', D)

        # Ghép [CLS] token và Positional Embedding rồi đưa qua TransformerEncoder
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, D)
        seq = torch.cat([cls_tokens, seq], dim=1)  # (B, T'+1, D)

        seq_len = seq.size(1)
        if seq_len <= self.pos_embedding.size(1):
            pos_emb = self.pos_embedding[:, :seq_len, :]
        else:
            pos_emb = F.interpolate(
                self.pos_embedding.permute(0, 2, 1),
                size=seq_len,
                mode='linear',
                align_corners=False
            ).permute(0, 2, 1)

        seq = seq + pos_emb

        # Xây dựng padding mask cho Transformer: True = bỏ qua (vùng pad), False = hợp lệ
        padding_mask = torch.zeros((B, seq_len), dtype=torch.bool, device=x.device)
        is_pad = (curr_mask.squeeze(1).squeeze(-1) <= 0.5)  # (B, T_prime) bool
        padding_mask[:, 1:1 + T_prime] = is_pad

        encoded = self.transformer(seq, src_key_padding_mask=padding_mask)  # (B, T'+1, D)
        cls_out = encoded[:, 0]  # Lấy đặc trưng tại [CLS] token

        return self.head(cls_out)

In [9]:
# Cell 7: Pipeline huấn luyện hướng đối tượng (OOP) Driver Guardian GCN (train.py - GCNTrainer)
# Lưu trữ toàn bộ các checkpoint của từng epoch đã chạy (epoch_XX.pt), best.pt và last.pt
# =====================================================================
# 1. TIỆN ÍCH HỆ THỐNG & HÀM ĐỘC LẬP (BACKWARD-COMPATIBLE HELPERS)
# =====================================================================
def seed_everything(seed: int = 42) -> None:
    """Cố định seed để đảm bảo khả năng tái lập kết quả thực nghiệm."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def configure_logging(log_file: Path | str) -> None:
    """Cấu hình logging đồng thời ra Console và file train.log."""
    if hasattr(sys.stdout, "reconfigure"):
        try:
            sys.stdout.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass

    log_file = Path(log_file)
    log_file.parent.mkdir(parents=True, exist_ok=True)

    logger.setLevel(logging.INFO)
    logger.propagate = False
    for handler in logger.handlers[:]:
        handler.close()
        logger.removeHandler(handler)

    formatter = logging.Formatter("[%(asctime)s][%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")

    file_handler = logging.FileHandler(log_file, encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)


def atomic_save(state: Any, path: str | Path) -> None:
    """Lưu tệp an toàn (Atomic Save qua file tạm) để tránh hỏng checkpoint khi bị ngắt đột ngột."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_file = tempfile.mkstemp(dir=path.parent, prefix=f".{path.name}.", suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            torch.save(state, f)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp_file, path)
    finally:
        Path(tmp_file).unlink(missing_ok=True)


def build_model(cfg: TrainConfig) -> nn.Module:
    """Khởi tạo mô hình dựa trên TrainConfig."""
    model_name = cfg.model_name.lower().strip()
    if model_name in ("stgcn", "driverstgcn", "light"):
        logger.info("Khởi tạo kiến trúc DriverSTGCN (ST-GCN + AttentiveDrowsinessOutputHead)...")
        return DriverSTGCN(**cfg.get_stgcn_kwargs())
    elif model_name in ("transformer", "driverstgcntransformer", "trans"):
        logger.info("Khởi tạo kiến trúc DriverSTGCNTransformer (STGCN + TransformerEncoder)...")
        return DriverSTGCNTransformer(**cfg.get_transformer_kwargs())
    else:
        raise ValueError(f"Không hỗ trợ model_name='{cfg.model_name}'. Chọn 'stgcn' hoặc 'transformer'.")


def get_cosine_warmup_scheduler(
        optimizer: torch.optim.Optimizer,
        total_epochs: int,
        warmup_epochs: int = 5,
        min_lr_ratio: float = 0.05,
) -> LambdaLR:
    """Tạo bộ điều tốc Cosine Annealing kèm giai đoạn Linear Warmup."""
    warmup_epochs = max(0, min(warmup_epochs, total_epochs - 1))

    def lr_lambda(epoch: int) -> float:
        if epoch < warmup_epochs:
            return float(epoch + 1) / float(max(1, warmup_epochs))
        progress = float(epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs - 1))
        progress = min(1.0, max(0.0, progress))
        return min_lr_ratio + (1.0 - min_lr_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress))

    return LambdaLR(optimizer, lr_lambda=lr_lambda)


def compute_metrics(
        y_true: np.ndarray,
        y_pred: np.ndarray,
        avg_loss: float,
        num_classes: int = 2,
) -> Dict[str, float]:
    """
    Tính toán chi tiết các chỉ số phân loại: Loss, Accuracy, Recall, F1-score, F2-score, Precision.
    - Với bài toán 2 lớp (0: Alert, 1: Drowsy): tập trung vào nhãn dương (1: Drowsy).
    - Với bài toán đa lớp (> 2 lớp): tính trung bình Macro trên tất cả các lớp.
    """
    total = len(y_true)
    if total == 0:
        return {
            "loss": float(avg_loss),
            "accuracy": 0.0,
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "f2": 0.0,
            "n_samples": 0,
        }

    accuracy = float((y_true == y_pred).sum() / total)

    if num_classes == 2:
        tp = int(((y_pred == 1) & (y_true == 1)).sum())
        fp = int(((y_pred == 1) & (y_true == 0)).sum())
        fn = int(((y_pred == 0) & (y_true == 1)).sum())

        precision = float(tp / max(1, tp + fp))
        recall = float(tp / max(1, tp + fn))
        f1 = float((2 * tp) / max(1, 2 * tp + fp + fn))
        f2 = float((5 * tp) / max(1, 5 * tp + 4 * fn + fp))
    else:
        precisions, recalls, f1s, f2s = [], [], [], []
        for c in range(num_classes):
            tp_c = int(((y_pred == c) & (y_true == c)).sum())
            fp_c = int(((y_pred == c) & (y_true != c)).sum())
            fn_c = int(((y_pred != c) & (y_true == c)).sum())

            p_c = tp_c / max(1, tp_c + fp_c)
            r_c = tp_c / max(1, tp_c + fn_c)
            f1_c = (2 * p_c * r_c) / max(1e-12, p_c + r_c)
            f2_c = (5 * p_c * r_c) / max(1e-12, 4 * p_c + r_c)

            precisions.append(p_c)
            recalls.append(r_c)
            f1s.append(f1_c)
            f2s.append(f2_c)

        precision = float(np.mean(precisions))
        recall = float(np.mean(recalls))
        f1 = float(np.mean(f1s))
        f2 = float(np.mean(f2s))

    return {
        "loss": float(avg_loss),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f2": f2,
        "n_samples": total,
    }


def run_epoch(
        model: nn.Module,
        loader: DataLoader,
        criterion: nn.Module,
        device: torch.device,
        optimizer: Optional[torch.optim.Optimizer] = None,
        scaler: Optional[torch.amp.GradScaler] = None,
        amp: bool = True,
        max_grad_norm: float = 1.0,
        desc: str = "Train",
        num_classes: int = 2,
) -> Dict[str, float]:
    """Chạy 1 epoch đầy đủ qua DataLoader và trả về bộ chỉ số tổng hợp."""
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_samples = 0
    all_targets = []
    all_preds = []

    progress = tqdm(loader, desc=desc, unit="batch", dynamic_ncols=True, leave=False)

    for batch in progress:
        landmarks, labels, lengths = batch[0], batch[1], batch[2]
        landmarks = landmarks.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        lengths = lengths.to(device, non_blocking=True)
        count = labels.size(0)

        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=device.type, enabled=amp and device.type == "cuda"):
                outputs = model(landmarks, lengths=lengths)
                logits = outputs["logits"] if isinstance(outputs, dict) else outputs
                loss = criterion(logits, labels)

            if not torch.isfinite(loss):
                raise FloatingPointError(f"Gặp giá trị Loss không hợp lệ (NaN/Inf) tại {desc}")

            if training:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None and scaler.is_enabled():
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    if max_grad_norm > 0:
                        nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    if max_grad_norm > 0:
                        nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
                    optimizer.step()

        preds = logits.detach().argmax(dim=1)
        total_loss += loss.detach().item() * count
        total_samples += count

        all_targets.append(labels.detach().cpu().numpy())
        all_preds.append(preds.cpu().numpy())

        current_loss = total_loss / max(1, total_samples)
        progress.set_postfix(loss=f"{current_loss:.4f}")

    if total_samples == 0:
        return {
            "loss": 0.0,
            "accuracy": 0.0,
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "f2": 0.0,
            "n_samples": 0,
        }

    y_true = np.concatenate(all_targets, axis=0)
    y_pred = np.concatenate(all_preds, axis=0)
    avg_loss = total_loss / total_samples

    return compute_metrics(y_true, y_pred, avg_loss, num_classes=num_classes)


def log_epoch_to_tensorboard(
        writer: SummaryWriter,
        train_metrics: Dict[str, float],
        val_metrics: Dict[str, float],
        current_lr: float,
        epoch: int,
) -> None:
    """Ghi nhận chi tiết các scalar Loss, Accuracy, Recall, F1, F2 cho cả Train & Val."""
    for metric_name in ("loss", "accuracy", "recall", "f1", "f2", "precision"):
        if metric_name in train_metrics:
            writer.add_scalar(f"train/{metric_name}", train_metrics[metric_name], epoch)
        if metric_name in val_metrics:
            writer.add_scalar(f"val/{metric_name}", val_metrics[metric_name], epoch)

    writer.add_scalar("train/learning_rate", current_lr, epoch)

    for metric_name in ("loss", "accuracy", "recall", "f1", "f2"):
        writer.add_scalars(
            f"compare/{metric_name}",
            {"train": train_metrics.get(metric_name, 0.0), "val": val_metrics.get(metric_name, 0.0)},
            epoch,
        )
    writer.flush()


# =====================================================================
# 2. LỚP QUẢN LÝ HUẤN LUYỆN HƯỚNG ĐỐI TƯỢNG (OOP GCN TRAINER)
# =====================================================================
class GCNTrainer:
    """
    Lớp Trainer hướng đối tượng (OOP) quản lý toàn bộ vòng đời huấn luyện mô hình Driver Guardian GCN:
    1. Khởi tạo và đồng bộ siêu tham số cấu hình (TrainConfig).
    2. Xây dựng DataLoaders (Train, Val, Test) tích hợp Cache .pt & InsightFace 68 3D Landmarks.
    3. Khởi tạo kiến trúc mô hình (DriverSTGCN / DriverSTGCNTransformer), Hàm mất mát, Optimizer, Scheduler và Scaler (AMP).
    4. Điều phối vòng lặp huấn luyện từng epoch, tính toán chỉ số chi tiết (Loss, Accuracy, Recall, F1, F2).
    5. Quản lý lưu trữ checkpoint nguyên tử (Atomic Save) cho từng epoch (epoch_XX.pt), best.pt và last.pt.
    6. Ghi nhận log trực quan theo thời gian thực ra Console, train.log và TensorBoard.
    7. Tự động đánh giá kiểm thử nghiệm thu trên tập Test bằng checkpoint tốt nhất sau khi kết thúc.
    """

    seed_everything = staticmethod(seed_everything)
    configure_logging = staticmethod(configure_logging)
    atomic_save = staticmethod(atomic_save)
    compute_metrics = staticmethod(compute_metrics)

    def __init__(self, cfg: Optional[TrainConfig] = None):
        self.cfg = cfg if cfg is not None else TrainConfig()
        self.cfg.__post_init__()

        self.save_dir = Path(self.cfg.save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)

        self.log_dir = Path(self.cfg.log_dir) if self.cfg.log_dir else (self.save_dir / "tensorboard")
        self.log_dir.mkdir(parents=True, exist_ok=True)

        self.configure_logging(self.save_dir / "train.log")
        self.seed_everything(self.cfg.seed)

        self.device = torch.device(self.cfg.device if torch.cuda.is_available() or self.cfg.device == "cpu" else "cpu")
        self.writer: Optional[SummaryWriter] = None

        self.cfg.save(self.save_dir / "config.json")
        self._setup_components()

    def _setup_components(self) -> None:
        """Khởi tạo DataLoaders, Mô hình, Hàm mất mát, Optimizer, Scheduler và Scaler."""
        logger.info("=" * 70)
        logger.info("KHỞI TẠO PIPELINE HUẤN LUYỆN GCNTRAINER (OOP) - DRIVER GUARDIAN GCN")
        logger.info("=" * 70)
        logger.info(f"Thiết bị sử dụng (Device)      : {self.device}")
        logger.info(f"Kiến trúc mô hình (Model)      : {self.cfg.model_name}")
        logger.info(f"Số lớp phân loại (Classes)     : {self.cfg.num_classes} ({self.cfg.class_names})")
        logger.info(f"Mô hình Landmark (InsightFace) : {self.cfg.landmark_model_name} (Det: {self.cfg.det_size})")
        logger.info(f"Sử dụng Cache Landmark (.pt)   : {self.cfg.use_cache} (Dir: {self.cfg.cache_dir})")
        logger.info(f"Hệ số Label Smoothing          : {self.cfg.label_smoothing}")
        logger.info(f"Thư mục lưu Checkpoint         : {self.save_dir}")
        logger.info(f"Thư mục log TensorBoard        : {self.log_dir}")

        self.train_loader, self.val_loader, self.test_loader = self._build_dataloaders()
        logger.info(
            f"Số lượng mẫu Dataset           : Train={len(self.train_loader.dataset)} | "
            f"Val={len(self.val_loader.dataset)} | Test={len(self.test_loader.dataset)}"
        )

        self.model = self._build_model().to(self.device)
        num_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        logger.info(f"Tổng số tham số học được       : {num_params:,} parameters")

        self.criterion = nn.CrossEntropyLoss(label_smoothing=self.cfg.label_smoothing)
        self.val_criterion = nn.CrossEntropyLoss()
        self.optimizer = AdamW(
            (p for p in self.model.parameters() if p.requires_grad),
            lr=self.cfg.lr,
            weight_decay=self.cfg.weight_decay,
        )

        self.scheduler = self._build_scheduler()
        self.scaler = torch.amp.GradScaler("cuda", enabled=self.cfg.amp and self.device.type == "cuda")

        self.start_epoch = 1
        self.best_epoch = 0
        self.best_metrics: Optional[Dict[str, float]] = None

        if self.cfg.resume:
            self.load_checkpoint(self.cfg.resume)

    def _build_dataloaders(self):
        return build_gcn_dataloaders(**self.cfg.get_dataloader_kwargs())

    def _build_model(self) -> nn.Module:
        return build_model(self.cfg)

    def _build_scheduler(self) -> LambdaLR:
        return get_cosine_warmup_scheduler(
            self.optimizer,
            total_epochs=self.cfg.epochs,
            warmup_epochs=self.cfg.warmup_epochs,
            min_lr_ratio=self.cfg.min_lr_ratio,
        )

    def load_checkpoint(self, resume_path: str | Path) -> None:
        """Khôi phục trạng thái mô hình và optimizer từ checkpoint đã lưu."""
        resume_path = Path(resume_path)
        if not resume_path.is_file():
            logger.warning(f"Không tìm thấy file checkpoint resume tại: {resume_path}. Bỏ qua nạp lại.")
            return

        logger.info(f"Nạp lại checkpoint từ: {resume_path}")
        try:
            checkpoint = torch.load(resume_path, map_location=self.device, weights_only=False)
        except TypeError:
            checkpoint = torch.load(resume_path, map_location=self.device)

        self.model.load_state_dict(checkpoint["model_state_dict"])
        if "optimizer_state_dict" in checkpoint:
            self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        if "scheduler_state_dict" in checkpoint:
            self.scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        if "scaler_state_dict" in checkpoint and self.scaler is not None and checkpoint["scaler_state_dict"] is not None:
            self.scaler.load_state_dict(checkpoint["scaler_state_dict"])
        self.start_epoch = checkpoint.get("epoch", 0) + 1
        self.best_epoch = checkpoint.get("best_epoch", 0)
        self.best_metrics = checkpoint.get("best_metrics", None)
        logger.info(f"Đã khôi phục thành công! Tiếp tục từ Epoch {self.start_epoch} (Best Epoch trước đó: {self.best_epoch})")

    def save_checkpoints(
            self,
            epoch: int,
            train_metrics: Dict[str, float],
            val_metrics: Dict[str, float],
            is_best: bool,
    ) -> None:
        """Lưu trữ an toàn mọi checkpoint của từng epoch (epoch_XX.pt), best.pt và last.pt."""
        checkpoint_state = {
            "format_version": 1,
            "epoch": epoch,
            "model_name": self.cfg.model_name,
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "scheduler_state_dict": self.scheduler.state_dict(),
            "scaler_state_dict": self.scaler.state_dict() if self.scaler else None,
            "train_metrics": train_metrics,
            "val_metrics": val_metrics,
            "best_epoch": self.best_epoch if not is_best else epoch,
            "best_metrics": self.best_metrics if not is_best else val_metrics,
            "config": self.cfg.to_dict(),
        }

        epoch_ckpt_path = self.save_dir / f"epoch_{epoch:02d}.pt"
        self.atomic_save(checkpoint_state, epoch_ckpt_path)
        self.atomic_save(checkpoint_state, self.save_dir / "last.pt")

        if is_best:
            self.best_epoch = epoch
            self.best_metrics = val_metrics
            best_state = {
                "format_version": 1,
                "epoch": self.best_epoch,
                "model_name": self.cfg.model_name,
                "model_state_dict": {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()},
                "train_metrics": train_metrics,
                "val_metrics": self.best_metrics,
                "config": self.cfg.to_dict(),
            }
            self.atomic_save(best_state, self.save_dir / "best.pt")
            with open(self.save_dir / "best_metrics.json", "w", encoding="utf-8") as f:
                json.dump(self.best_metrics, f, indent=4, ensure_ascii=False)

    def log_epoch_to_tensorboard(
            self,
            train_metrics: Dict[str, float],
            val_metrics: Dict[str, float],
            current_lr: float,
            epoch: int,
    ) -> None:
        if self.writer is None:
            return
        log_epoch_to_tensorboard(self.writer, train_metrics, val_metrics, current_lr, epoch)

    def run_epoch(
            self,
            loader: DataLoader,
            training: bool = True,
            desc: str = "Train",
    ) -> Dict[str, float]:
        """Chạy 1 epoch đầy đủ qua DataLoader (dùng cho cả Train, Val, Test)."""
        return run_epoch(
            model=self.model,
            loader=loader,
            criterion=self.criterion if training else self.val_criterion,
            device=self.device,
            optimizer=self.optimizer if training else None,
            scaler=self.scaler if training else None,
            amp=self.cfg.amp,
            max_grad_norm=self.cfg.max_grad_norm,
            desc=desc,
            num_classes=self.cfg.num_classes,
        )

    def train_one_epoch(self, epoch: int) -> Dict[str, float]:
        if hasattr(self.model, "output_head") and hasattr(self.model.output_head, "grl"):
            progress = float(epoch - 1) / float(max(1, self.cfg.epochs - 1))
            self.model.output_head.grl.set_lambda(progress)

        return self.run_epoch(
            loader=self.train_loader,
            training=True,
            desc=f"Epoch {epoch:02d}/{self.cfg.epochs:02d} [Train]",
        )

    def validate(self, epoch: int) -> Dict[str, float]:
        return self.run_epoch(
            loader=self.val_loader,
            training=False,
            desc=f"Epoch {epoch:02d}/{self.cfg.epochs:02d} [Val]",
        )

    def evaluate_test(self) -> Optional[Dict[str, float]]:
        """Đánh giá kiểm thử nghiệm thu trên tập Test bằng checkpoint tốt nhất (best.pt)."""
        best_pt_path = self.save_dir / "best.pt"
        if not best_pt_path.is_file() or len(self.test_loader.dataset) == 0:
            return None

        logger.info("\nĐang tải best.pt để đánh giá nghiệm thu trên tập Test...")
        try:
            checkpoint = torch.load(best_pt_path, map_location=self.device, weights_only=False)
        except TypeError:
            checkpoint = torch.load(best_pt_path, map_location=self.device)
        self.model.load_state_dict(checkpoint["model_state_dict"])

        test_metrics = self.run_epoch(
            loader=self.test_loader,
            training=False,
            desc="Final Test",
        )

        if self.writer is not None:
            for m_name, m_val in test_metrics.items():
                self.writer.add_scalar(f"test/{m_name}", m_val, self.best_epoch)
            self.writer.flush()

        with open(self.save_dir / "test_metrics.json", "w", encoding="utf-8") as f:
            json.dump(test_metrics, f, indent=4, ensure_ascii=False)

        logger.info(
            f"Kết quả Test (Best Epoch {self.best_epoch}): "
            f"Loss={test_metrics['loss']:.4f} | Acc={test_metrics['accuracy']:.4f} | "
            f"Recall={test_metrics['recall']:.4f} | F1={test_metrics['f1']:.4f} | F2={test_metrics['f2']:.4f}"
        )
        return test_metrics

    def fit(self) -> nn.Module:
        """Thực thi toàn bộ quy trình huấn luyện và đánh giá mô hình."""
        self.writer = SummaryWriter(str(self.log_dir))
        self.writer.add_text(
            "Config",
            f"```json\n{json.dumps(self.cfg.to_dict(), indent=2, ensure_ascii=False)}\n```",
            0
        )

        logger.info("=" * 70)
        logger.info("BẮT ĐẦU VÒNG LẶP HUẤN LUYỆN (FIT LOOP)")
        logger.info("=" * 70)

        start_train_time = time.time()

        try:
            for epoch in range(self.start_epoch, self.cfg.epochs + 1):
                epoch_start = time.time()
                current_lr = float(self.optimizer.param_groups[0]["lr"])

                train_metrics = self.train_one_epoch(epoch)
                val_metrics = self.validate(epoch)

                self.scheduler.step()
                epoch_duration = time.time() - epoch_start

                self.log_epoch_to_tensorboard(train_metrics, val_metrics, current_lr, epoch)

                is_best = False
                if self.best_metrics is None:
                    is_best = True
                else:
                    val_score = (val_metrics["f1"], val_metrics["f2"], -val_metrics["loss"])
                    best_score = (self.best_metrics["f1"], self.best_metrics["f2"], -self.best_metrics["loss"])
                    if val_score > best_score:
                        is_best = True

                self.save_checkpoints(epoch, train_metrics, val_metrics, is_best)

                best_tag = " >>> [BEST RECORD UPDATED] <<<" if is_best else ""
                logger.info(
                    f"Epoch {epoch:02d}/{self.cfg.epochs:02d} ({epoch_duration:.1f}s) | LR: {current_lr:.6g} [Checkpoint: epoch_{epoch:02d}.pt]\n"
                    f"  - Train: Loss={train_metrics['loss']:.4f} | Acc={train_metrics['accuracy']:.4f} | Recall={train_metrics['recall']:.4f} | F1={train_metrics['f1']:.4f} | F2={train_metrics['f2']:.4f}\n"
                    f"  - Val  : Loss={val_metrics['loss']:.4f} | Acc={val_metrics['accuracy']:.4f} | Recall={val_metrics['recall']:.4f} | F1={val_metrics['f1']:.4f} | F2={val_metrics['f2']:.4f}{best_tag}"
                )

        except KeyboardInterrupt:
            logger.warning("\nQuá trình huấn luyện bị ngắt bởi người dùng. Toàn bộ checkpoint đã được bảo lưu an toàn.")
        finally:
            total_time = time.time() - start_train_time
            logger.info(f"\nHoàn tất huấn luyện sau {total_time / 60:.2f} phút.")
            if self.best_metrics:
                logger.info(f"Checkpoint tốt nhất tại Epoch {self.best_epoch}:")
                logger.info(f"  - Val F1      : {self.best_metrics['f1']:.4f}")
                logger.info(f"  - Val F2      : {self.best_metrics['f2']:.4f}")
                logger.info(f"  - Val Recall  : {self.best_metrics['recall']:.4f}")
                logger.info(f"  - Val Accuracy: {self.best_metrics['accuracy']:.4f}")
                logger.info(f"  - Val Loss    : {self.best_metrics['loss']:.4f}")

            self.evaluate_test()

            if self.writer is not None:
                self.writer.close()

        return self.model


def train(cfg: Optional[TrainConfig] = None) -> nn.Module:
    """Hàm thực thi huấn luyện toàn bộ pipeline thông qua GCNTrainer."""
    trainer = GCNTrainer(cfg)
    return trainer.fit()

In [10]:
# Cell 8: Khởi tạo cấu hình TrainConfig (mặc định DriverSTGCN), đối tượng GCNTrainer và bắt đầu huấn luyện
config = TrainConfig(model_name="stgcn")
trainer = GCNTrainer(config)
trained_model = trainer.fit()

[2026-11-03 10:01:06][INFO] ======================================================================
[2026-11-03 10:01:06][INFO] KHỞI TẠO PIPELINE HUẤN LUYỆN GCNTRAINER (OOP) - DRIVER GUARDIAN GCN
[2026-11-03 10:01:06][INFO] ======================================================================
[2026-11-03 10:01:06][INFO] Thiết bị sử dụng (Device)      : cuda
[2026-11-03 10:01:06][INFO] Kiến trúc mô hình (Model)      : stgcn
[2026-11-03 10:01:06][INFO] Số lớp phân loại (Classes)     : 2 ({0: 'Alert', 1: 'Drowsy'})
[2026-11-03 10:01:06][INFO] Mô hình Landmark (InsightFace) : buffalo_l (Det: (320, 320))
[2026-11-03 10:01:06][INFO] Sử dụng Cache Landmark (.pt)   : True (Dir: /kaggle/input/cache-dataset-processed-v3/cache_landmarks)
[2026-11-03 10:01:06][INFO] Hệ số Label Smoothing          : 0.0
[2026-11-03 10:01:06][INFO] Thư mục lưu Checkpoint         : GCN/checkpoints
[2026-11-03 10:01:06][INFO] Thư mục log TensorBoard        : GCN/checkpoints/tensorboard
[2026-11-03 10:01:07][INFO] [CAC

[2026-11-03 10:02:23][INFO] Epoch 01/20 (74.6s) | LR: 0.0002 [Checkpoint: epoch_01.pt]
  - Train: Loss=0.6570 | Acc=0.6181 | Recall=0.5560 | F1=0.5824 | F2=0.5662
  - Val  : Loss=0.6230 | Acc=0.6661 | Recall=0.5543 | F1=0.6152 | F2=0.5772 >>> [BEST RECORD UPDATED] <<<


[2026-11-03 10:03:36][INFO] Epoch 02/20 (73.3s) | LR: 0.0004 [Checkpoint: epoch_02.pt]
  - Train: Loss=0.6162 | Acc=0.6592 | Recall=0.5632 | F1=0.6128 | F2=0.5820
  - Val  : Loss=0.5734 | Acc=0.6991 | Recall=0.4857 | F1=0.6086 | F2=0.5284


[2026-11-03 10:04:50][INFO] Epoch 03/20 (73.4s) | LR: 0.0006 [Checkpoint: epoch_03.pt]
  - Train: Loss=0.5825 | Acc=0.6973 | Recall=0.6358 | F1=0.6679 | F2=0.6483
  - Val  : Loss=0.7171 | Acc=0.6844 | Recall=0.3981 | F1=0.5486 | F2=0.4472


[2026-11-03 10:06:04][INFO] Epoch 04/20 (73.4s) | LR: 0.0008 [Checkpoint: epoch_04.pt]
  - Train: Loss=0.5452 | Acc=0.7234 | Recall=0.6992 | F1=0.7077 | F2=0.7025
  - Val  : Loss=0.7580 | Acc=0.7202 | Recall=0.5619 | F1=0.6592 | F2=0.5972 >>> [BEST RECORD UPDATED] <<<


[2026-11-03 10:07:17][INFO] Epoch 05/20 (73.5s) | LR: 0.001 [Checkpoint: epoch_05.pt]
  - Train: Loss=0.5225 | Acc=0.7486 | Recall=0.7312 | F1=0.7358 | F2=0.7331
  - Val  : Loss=0.6961 | Acc=0.6642 | Recall=0.5067 | F1=0.5924 | F2=0.5378


... (12 outputs left out)

[2026-11-03 10:23:13][INFO] Epoch 18/20 (73.4s) | LR: 9.70398e-05 [Checkpoint: epoch_18.pt]
  - Train: Loss=0.2276 | Acc=0.9045 | Recall=0.8973 | F1=0.9000 | F2=0.8984
  - Val  : Loss=1.2973 | Acc=0.6294 | Recall=0.4000 | F1=0.5097 | F2=0.4377


[2026-11-03 10:24:27][INFO] Epoch 19/20 (73.5s) | LR: 6.19092e-05 [Checkpoint: epoch_19.pt]
  - Train: Loss=0.2239 | Acc=0.9059 | Recall=0.8985 | F1=0.9014 | F2=0.8997
  - Val  : Loss=1.2670 | Acc=0.6376 | Recall=0.4000 | F1=0.5153 | F2=0.4393


[2026-11-03 10:25:41][INFO] Epoch 20/20 (73.6s) | LR: 5e-05 [Checkpoint: epoch_20.pt]
  - Train: Loss=0.2142 | Acc=0.9155 | Recall=0.9093 | F1=0.9115 | F2=0.9102
  - Val  : Loss=1.3841 | Acc=0.6156 | Recall=0.3867 | F1=0.4921 | F2=0.4229
[2026-11-03 10:25:41][INFO] 
Hoàn tất huấn luyện sau 24.54 phút.
[2026-11-03 10:25:41][INFO] Checkpoint tốt nhất tại Epoch 6:
[2026-11-03 10:25:41][INFO]   - Val F1      : 0.7282
[2026-11-03 10:25:41][INFO]   - Val F2      : 0.7057
[2026-11-03 10:25:41][INFO]   - Val Recall  : 0.6914
[2026-11-03 10:25:41][INFO]   - Val Accuracy: 0.7514
[2026-11-03 10:25:41][INFO]   - Val Loss    : 0.5958
[2026-11-03 10:25:41][INFO] 
Đang tải best.pt để đánh giá nghiệm thu trên tập Test...


[2026-11-03 10:25:45][INFO] Kết quả Test (Best Epoch 6): Loss=0.8756 | Acc=0.6041 | Recall=0.5992 | F1=0.5974 | F2=0.5985


In [12]:
!zip -r hazards_results.zip GCN


  adding: GCN/ (stored 0%)
  adding: GCN/checkpoints/ (stored 0%)
  adding: GCN/checkpoints/config.json (deflated 71%)
  adding: GCN/checkpoints/epoch_09.pt (deflated 9%)
  adding: GCN/checkpoints/epoch_17.pt (deflated 9%)
  adding: GCN/checkpoints/epoch_15.pt (deflated 9%)
  adding: GCN/checkpoints/test_metrics.json (deflated 38%)
  adding: GCN/checkpoints/epoch_19.pt (deflated 9%)
  adding: GCN/checkpoints/tensorboard/ (stored 0%)
  adding: GCN/checkpoints/tensorboard/compare_accuracy_train/ (stored 0%)
  adding: GCN/checkpoints/tensorboard/compare_accuracy_train/events.out.tfevents.1793700143.982b2d44c3cb.14366.3 (deflated 57%)
  adding: GCN/checkpoints/tensorboard/compare_f1_val/ (stored 0%)
  adding: GCN/checkpoints/tensorboard/compare_f1_val/events.out.tfevents.1793700143.982b2d44c3cb.14366.8 (deflated 53%)
  adding: GCN/checkpoints/tensorboard/compare_f2_val/ (stored 0%)
  adding: GCN/checkpoints/tensorboard/compare_f2_val/events.out.tfevents.1793700143.982b2d44c3cb.14366.10 (de